# 🍄 Gljive Hrvatske – fotografije za atlas

Za svaku vrstu bez fotografije uzima **glavnu fotografiju vrste s iNaturalista** (onu koju zajednica postavi kao reprezentativnu), a ako je nema, najbolje ocijenjenu provjerenu fotografiju nalaza ili fotografiju s **Wikimedia Commonsa**. Uzima samo licence **CC0, CC BY i CC BY-SA** (dopuštaju i komercijalnu upotrebu uz navođenje autora).

**Pokretanje:** Runtime → **Run all**. GPU nije potreban. Traje oko 20–30 minuta.
Na kraju se skine **atlas_foto.zip** (oko 20 MB), a kopija ostaje na Google Driveu u mapi *GljiveModel*. Taj zip pošalji Claudeu.

In [ ]:
import json, time, io, re, zipfile, html as _html
from pathlib import Path
import requests
from PIL import Image
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = Path('/content/drive/MyDrive/GljiveModel')
except ImportError:
    OUT = Path('GljiveModel')
OUT.mkdir(parents=True, exist_ok=True)
WORK = Path('/content/foto' if Path('/content').exists() else 'foto_work'); WORK.mkdir(exist_ok=True)
S = requests.Session(); S.headers['User-Agent'] = 'GljiveHrvatske-atlas/1.0 (github.com/markofrankovic-afk/Gljive-Hrvatske)'
API = 'https://api.inaturalist.org/v1/'
LIC_OK = ['cc0', 'cc-by', 'cc-by-sa']
LIC_TXT = {'cc0': 'CC0', 'cc-by': 'CC BY', 'cc-by-sa': 'CC BY-SA'}
def api(path, **params):
    for k in range(5):
        try:
            r = S.get(API + path, params=params, timeout=60); time.sleep(0.7)
            if r.status_code == 200: return r.json()
            if r.status_code in (429, 500, 502, 503, 504): time.sleep(20 * (k + 1)); continue
            return None
        except requests.RequestException:
            time.sleep(15 * (k + 1))
    return None
TAKS_F = OUT / 'taksoni.json'
TAKS = json.loads(TAKS_F.read_text()) if TAKS_F.exists() else {}
print('Spremljenih taksona s Drivea:', len(TAKS))

In [ ]:
TREBA = json.loads('[{"id": "zucara", "lat": "Tylopilus felleus", "hr": "Žučara", "shape": "convex", "nc": true}, {"id": "ludara", "lat": "Rubroboletus satanas", "hr": "Ludara", "shape": "convex", "nc": true}, {"id": "lisicarka", "lat": "Cantharellus cibarius", "hr": "Lisičarka", "shape": "funnel", "nc": true}, {"id": "lazna-lisicarka", "lat": "Hygrophoropsis aurantiaca", "hr": "Lažna lisičarka", "shape": "funnel", "nc": true}, {"id": "omphalotus", "lat": "Omphalotus olearius", "hr": "Zavodnica", "shape": "funnel", "nc": true}, {"id": "pupavka", "lat": "Amanita phalloides", "hr": "Zelena pupavka", "shape": "convex", "nc": true}, {"id": "bijela-pupavka", "lat": "Amanita virosa", "hr": "Bijela pupavka", "shape": "conical", "nc": true}, {"id": "muhara", "lat": "Amanita muscaria", "hr": "Crvena muhara", "shape": "convex", "nc": true}, {"id": "panterovka", "lat": "Amanita pantherina", "hr": "Panterova muhara", "shape": "convex", "nc": true}, {"id": "biserka", "lat": "Amanita rubescens", "hr": "Biserna muhara", "shape": "convex", "nc": true}, {"id": "smrcak", "lat": "Morchella esculenta", "hr": "Smrčak", "shape": "morel", "nc": true}, {"id": "hrcak", "lat": "Gyromitra esculenta", "hr": "Hrčak", "shape": "brain", "nc": true}, {"id": "bukovaca", "lat": "Pleurotus ostreatus", "hr": "Bukovača", "shape": "shelf", "nc": true}, {"id": "sumporaca", "lat": "Hypholoma fasciculare", "hr": "Sumporača", "shape": "convex", "nc": true}, {"id": "pecurka", "lat": "Agaricus campestris", "hr": "Poljska pečurka", "shape": "convex", "nc": true}, {"id": "karbolna", "lat": "Agaricus xanthodermus", "hr": "Karbolna pečurka", "shape": "convex", "nc": true}, {"id": "krasnica-zelena", "lat": "Russula virescens", "hr": "Krastava krasnica", "shape": "flat", "nc": true}, {"id": "ljuta-krasnica", "lat": "Russula emetica", "hr": "Paprena krasnica", "shape": "flat", "nc": true}, {"id": "rujnica", "lat": "Lactarius deliciosus", "hr": "Plemenita rujnica", "shape": "funnel", "nc": true}, {"id": "trubaca", "lat": "Craterellus cornucopioides", "hr": "Mrka trubača", "shape": "funnel", "nc": true}, {"id": "tartuf", "lat": "Tuber magnatum", "hr": "Bijeli tartuf", "shape": "tuber", "nc": true}, {"id": "ljeponogi-vrganj", "lat": "Caloboletus calopus", "hr": "Ljeponogi vrganj", "shape": "convex", "nc": true}, {"id": "zelenjaca", "lat": "Neoboletus erythropus", "hr": "Zelenjača", "shape": "convex", "nc": true}, {"id": "kovara", "lat": "Suillellus luridus", "hr": "Kovara", "shape": "convex", "nc": true}, {"id": "glatkonoska", "lat": "Suillellus queletii", "hr": "Glatkonoška", "shape": "convex", "nc": true}, {"id": "kostanjevka", "lat": "Imleria badia", "hr": "Kostanjevka", "shape": "convex", "nc": true}, {"id": "zlataca", "lat": "Xerocomellus chrysenteron", "hr": "Zlatača", "shape": "convex", "nc": true}, {"id": "kestenjasti-vrganj", "lat": "Gyroporus castaneus", "hr": "Kestenjasti vrganj", "shape": "convex", "nc": true}, {"id": "modri-vrganj", "lat": "Gyroporus cyanescens", "hr": "Modri vrganj", "shape": "convex", "nc": true}, {"id": "brezov-djed", "lat": "Leccinum scabrum", "hr": "Brezov djed", "shape": "convex", "nc": true}, {"id": "crveni-hrapavac", "lat": "Leccinum aurantiacum", "hr": "Crveni hrapavac", "shape": "convex", "nc": true}, {"id": "plemeniti-hrapavac", "lat": "Leccinellum crocipodium", "hr": "Plemeniti hrapavac", "shape": "convex", "nc": true}, {"id": "zrnati-sluzavac", "lat": "Suillus granulatus", "hr": "Zrnati sluzavac", "shape": "convex", "nc": true}, {"id": "arisev-sluzavac", "lat": "Suillus grevillei", "hr": "Arišev sluzavac", "shape": "convex", "nc": true}, {"id": "pjescani-sluzavac", "lat": "Suillus variegatus", "hr": "Pješčani sluzavac", "shape": "convex", "nc": true}, {"id": "veliki-slinavac", "lat": "Gomphidius glutinosus", "hr": "Veliki slinavac", "shape": "convex", "nc": true}, {"id": "ljevkasta-lisicica", "lat": "Craterellus tubaeformis", "hr": "Ljevkasta lisičica", "shape": "funnel", "nc": true}, {"id": "zlatonoga-trubaca", "lat": "Craterellus lutescens", "hr": "Zlatonoga trubača", "shape": "funnel", "nc": true}, {"id": "bjelonoga-stitarka", "lat": "Lepiota clypeolaria", "hr": "Bjelonoga štitarka", "shape": "umbrella-s", "nc": true}, {"id": "golema-pecurka", "lat": "Agaricus augustus", "hr": "Golema pečurka", "shape": "convex", "nc": true}, {"id": "mala-sumarica", "lat": "Agaricus sylvaticus", "hr": "Mala šumarica", "shape": "convex", "nc": true}, {"id": "zuckasta-pupavka", "lat": "Amanita citrina", "hr": "Žućkasta pupavka", "shape": "convex", "nc": true}, {"id": "sarena-gnojistarka", "lat": "Coprinopsis picacea", "hr": "Šarena gnojištarka", "shape": "tall", "nc": true}, {"id": "siva-maglenka", "lat": "Clitocybe nebularis", "hr": "Siva maglenka", "shape": "convex", "nc": true}, {"id": "velika-martincica", "lat": "Infundibulicybe geotropa", "hr": "Velika martinčica", "shape": "funnel", "nc": true}, {"id": "krhka-brasnjaca", "lat": "Clitopilus prunulus", "hr": "Krhka brašnjača", "shape": "funnel", "nc": true}, {"id": "blijedoljubicasta-koturnica", "lat": "Lepista glaucocana", "hr": "Blijedoljubičasta koturnica", "shape": "convex", "nc": true}, {"id": "vilin-klincac", "lat": "Marasmius oreades", "hr": "Vilin klinčac", "shape": "convex", "nc": true}, {"id": "sivka", "lat": "Tricholoma portentosum", "hr": "Sivka", "shape": "convex", "nc": true}, {"id": "sapunasta-vitezovka", "lat": "Tricholoma saponaceum", "hr": "Sapunasta vitezovka", "shape": "convex", "nc": true}, {"id": "zutozelena-vitezovka", "lat": "Tricholoma sejunctum", "hr": "Žutozelena vitezovka", "shape": "convex", "nc": true}, {"id": "sjenovita-vitezovka", "lat": "Tricholoma sciodes", "hr": "Sjenovita vitezovka", "shape": "convex", "nc": true}, {"id": "blistava-zutoliska", "lat": "Tricholomopsis rutilans", "hr": "Blistava žutoliska", "shape": "convex", "nc": true}, {"id": "obicna-gustoliska", "lat": "Melanoleuca melaleuca", "hr": "Obična gustoliska", "shape": "convex", "nc": true}, {"id": "stoglava-srastenica", "lat": "Lyophyllum decastes", "hr": "Stoglava sraštenica", "shape": "convex", "nc": true}, {"id": "tvrdokozna-srastenica", "lat": "Lyophyllum loricatum", "hr": "Tvrdokožna sraštenica", "shape": "convex", "nc": true}, {"id": "bijela-srastenica", "lat": "Leucocybe connata", "hr": "Bijela sraštenica", "shape": "convex", "nc": true}, {"id": "orijaska-rupicarka", "lat": "Meripilus giganteus", "hr": "Orijaška rupičarka", "shape": "shelf", "nc": true}, {"id": "mekana-rupicarka", "lat": "Hapalopilus rutilans", "hr": "Mekana rupičarka", "shape": "shelf", "nc": true}, {"id": "judino-uho", "lat": "Auricularia auricula-judae", "hr": "Judino uho", "shape": "ear", "nc": true}, {"id": "barsunasta-panjevcica", "lat": "Flammulina velutipes", "hr": "Baršunasta panjevčica", "shape": "convex", "nc": true}, {"id": "vodenasta-panjevcica", "lat": "Psathyrella piluliformis", "hr": "Vodenasta panjevčica", "shape": "convex", "nc": true}, {"id": "jablanovaca", "lat": "Cyclocybe cylindracea", "hr": "Jablanovača", "shape": "convex", "nc": true}, {"id": "grmaca", "lat": "Desarmillaria tabescens", "hr": "Grmača", "shape": "convex", "nc": true}, {"id": "brezina-guba", "lat": "Fomitopsis betulina", "hr": "Brezina guba", "shape": "shelf", "nc": true}, {"id": "sluzava-korjenjaca", "lat": "Oudemansiella mucida", "hr": "Sluzava korjenjača", "shape": "convex", "nc": true}, {"id": "maslacna-plosnatica", "lat": "Rhodocollybia butyracea", "hr": "Maslačna plosnatica", "shape": "convex", "nc": true}, {"id": "crvenkasta-gljivica", "lat": "Laccaria laccata", "hr": "Crvenkasta gljivica", "shape": "convex", "nc": true}, {"id": "kosoglava-sljemovka", "lat": "Mycena inclinata", "hr": "Kosoglava šljemovka", "shape": "small", "nc": true}, {"id": "prijesna-mlijecnica", "lat": "Lactifluus volemus", "hr": "Prijesna mliječnica", "shape": "convex", "nc": true}, {"id": "ljubicastozelena-krasnica", "lat": "Russula cyanoxantha", "hr": "Ljubičastozelena krasnica", "shape": "flat", "nc": true}, {"id": "jestiva-krasnica", "lat": "Russula vesca", "hr": "Jestiva krasnica", "shape": "flat", "nc": true}, {"id": "smedastozelena-krasnica", "lat": "Russula heterophila", "hr": "Smeđastozelena krasnica", "shape": "flat", "nc": false}, {"id": "romellijeva-krasnica", "lat": "Russula romellii", "hr": "Romellijeva krasnica", "shape": "flat", "nc": true}, {"id": "modrolisna-krasnica", "lat": "Russula delica", "hr": "Modrolisna krasnica", "shape": "funnel", "nc": true}, {"id": "tvrda-krasnica", "lat": "Russula lepida", "hr": "Tvrda krasnica", "shape": "flat", "nc": true}, {"id": "maslinasta-krasnica", "lat": "Russula olivacea", "hr": "Maslinasta krasnica", "shape": "flat", "nc": true}, {"id": "cigancek", "lat": "Cortinarius caperatus", "hr": "Ciganček", "shape": "convex", "nc": true}, {"id": "ljuskava-koprenka", "lat": "Cortinarius bolaris", "hr": "Ljuskava koprenka", "shape": "convex", "nc": true}, {"id": "polukrvna-koprenka", "lat": "Cortinarius semisanguineus", "hr": "Polukrvna koprenka", "shape": "convex", "nc": true}, {"id": "obicna-koprenka", "lat": "Cortinarius trivialis", "hr": "Obična koprenka", "shape": "convex", "nc": true}, {"id": "siljasta-koprenka", "lat": "Cortinarius rubellus", "hr": "Šiljasta koprenka", "shape": "conical", "nc": true}, {"id": "blijedolisticava-koprenka", "lat": "Leucocortinarius bulbiger", "hr": "Blijedolistićava koprenka", "shape": "convex", "nc": true}, {"id": "usiljena-cjepaca", "lat": "Pseudosperma rimosum", "hr": "Ušiljena cjepača", "shape": "conical", "nc": true}, {"id": "gorusicina-plavulja", "lat": "Hebeloma sinapizans", "hr": "Gorušičina plavulja", "shape": "convex", "nc": true}, {"id": "obicna-uvijaca", "lat": "Paxillus involutus", "hr": "Obična uvijača", "shape": "funnel", "nc": true}, {"id": "barsunasta-uvijaca", "lat": "Tapinella atrotomentosa", "hr": "Baršunasta uvijača", "shape": "convex", "nc": true}, {"id": "sivkasta-sljivovaca", "lat": "Entoloma clypeatum", "hr": "Sivkasta šljivovača", "shape": "convex", "nc": true}, {"id": "ozujska-puzevica", "lat": "Hygrophorus marzuolus", "hr": "Ožujska puževica", "shape": "convex", "nc": true}, {"id": "medena-puzevica", "lat": "Hygrophorus russula", "hr": "Medena puževica", "shape": "convex", "nc": true}, {"id": "zelena-viticarka", "lat": "Stropharia aeruginosa", "hr": "Zelena vitičarka", "shape": "convex", "nc": true}, {"id": "ferijeva-viticarka", "lat": "Stropharia rugosoannulata", "hr": "Ferijeva vitičarka", "shape": "convex", "nc": true}, {"id": "crvenkasta-capica", "lat": "Ramaria botrytis", "hr": "Crvenkasta capica", "shape": "coral", "nc": true}, {"id": "otrovna-capica", "lat": "Ramaria formosa", "hr": "Otrovna capica", "shape": "coral", "nc": true}, {"id": "tikvasta-puhara", "lat": "Lycoperdon perlatum", "hr": "Tikvasta puhara", "shape": "ball", "nc": true}, {"id": "kruskasta-puhara", "lat": "Apioperdon pyriforme", "hr": "Kruškasta puhara", "shape": "ball", "nc": true}, {"id": "divovska-puhara", "lat": "Calvatia gigantea", "hr": "Divovska puhara", "shape": "ball", "nc": true}, {"id": "krumpiraca", "lat": "Scleroderma citrinum", "hr": "Obična krumpirača", "shape": "ball", "nc": true}, {"id": "siljasti-smrcak", "lat": "Morchella elata", "hr": "Šiljasti smrčak", "shape": "morel", "nc": true}, {"id": "ljetni-tartuf", "lat": "Tuber aestivum", "hr": "Ljetni crni tartuf", "shape": "tuber", "nc": true}, {"id": "x-hygrocybe-ceracea", "lat": "Hygrocybe ceracea", "hr": "Hygrocybe ceracea", "shape": "conical", "nc": true}, {"id": "x-boletus-ferrugineus", "lat": "Boletus ferrugineus", "hr": "Boletus ferrugineus", "shape": "convex", "nc": false}, {"id": "x-hygrocybe-acutoconica", "lat": "Hygrocybe acutoconica", "hr": "Hygrocybe acutoconica", "shape": "conical", "nc": true}, {"id": "x-cortinarius-largus", "lat": "Cortinarius largus", "hr": "Cortinarius largus", "shape": "convex", "nc": false}, {"id": "x-cortinarius-malicorius", "lat": "Cortinarius malicorius", "hr": "Cortinarius malicorius", "shape": "convex", "nc": true}, {"id": "x-cortinarius-torvus", "lat": "Cortinarius torvus", "hr": "Tamna koprenka", "shape": "convex", "nc": true}, {"id": "x-cortinarius-anserinus", "lat": "Cortinarius anserinus", "hr": "Cortinarius anserinus", "shape": "convex", "nc": false}, {"id": "x-mycena-leptocephala", "lat": "Mycena leptocephala", "hr": "Mycena leptocephala", "shape": "small", "nc": false}, {"id": "x-crucibulum-crucibuliforme", "lat": "Crucibulum crucibuliforme", "hr": "Crucibulum crucibuliforme", "shape": "small", "nc": false}, {"id": "x-hypholoma-marginatum", "lat": "Hypholoma marginatum", "hr": "Ljuskastonoga sumporača", "shape": "convex", "nc": true}, {"id": "x-mycena-polygramma", "lat": "Mycena polygramma", "hr": "Žljebonoga šljemovka", "shape": "small", "nc": false}, {"id": "x-russula-depallens", "lat": "Russula depallens", "hr": "Russula depallens", "shape": "flat", "nc": false}, {"id": "x-hygrocybe-insipida", "lat": "Hygrocybe insipida", "hr": "Hygrocybe insipida", "shape": "conical", "nc": true}, {"id": "x-mycena-pseudocorticola", "lat": "Mycena pseudocorticola", "hr": "Mycena pseudocorticola", "shape": "small", "nc": false}, {"id": "x-clitocybe-metachroa", "lat": "Clitocybe metachroa", "hr": "Clitocybe metachroa", "shape": "funnel", "nc": false}, {"id": "x-russula-praetervisa", "lat": "Russula praetervisa", "hr": "Russula praetervisa", "shape": "flat", "nc": true}, {"id": "x-gymnopus-aquosus", "lat": "Gymnopus aquosus", "hr": "Gymnopus aquosus", "shape": "flat", "nc": true}, {"id": "x-pluteus-phlebophorus", "lat": "Pluteus phlebophorus", "hr": "Pluteus phlebophorus", "shape": "convex", "nc": true}, {"id": "x-pluteus-romellii", "lat": "Pluteus romellii", "hr": "Pluteus romellii", "shape": "convex", "nc": true}, {"id": "x-coltricia-perennis", "lat": "Coltricia perennis", "hr": "Coltricia perennis", "shape": "funnel", "nc": false}, {"id": "x-chrysomphalina-grossula", "lat": "Chrysomphalina grossula", "hr": "Chrysomphalina grossula", "shape": "funnel", "nc": true}, {"id": "x-clitopilus-hobsonii", "lat": "Clitopilus hobsonii", "hr": "Clitopilus hobsonii", "shape": "shelf", "nc": false}, {"id": "x-inocybe-sindonia", "lat": "Inocybe sindonia", "hr": "Inocybe sindonia", "shape": "conical", "nc": true}, {"id": "x-mycena-abramsii", "lat": "Mycena abramsii", "hr": "Mycena abramsii", "shape": "small", "nc": true}, {"id": "x-conocybe-aporos", "lat": "Conocybe aporos", "hr": "Conocybe aporos", "shape": "conical", "nc": false}, {"id": "x-flammulina-elastica", "lat": "Flammulina elastica", "hr": "Flammulina elastica", "shape": "convex", "nc": true}, {"id": "x-cortinarius-obtusus", "lat": "Cortinarius obtusus", "hr": "Cortinarius obtusus", "shape": "convex", "nc": true}, {"id": "b-amanita-aureola", "lat": "Amanita aureola", "hr": "Ogoljela muhara", "shape": "convex", "nc": false}, {"id": "b-amanita-eliae", "lat": "Amanita eliae", "hr": "Elijeva muhara", "shape": "convex", "nc": false}, {"id": "b-amanita-beckeri", "lat": "Amanita beckeri", "hr": "Bekerova muhara", "shape": "convex", "nc": false}, {"id": "b-amanita-spissa", "lat": "Amanita spissa", "hr": "Prava tigrica", "shape": "convex", "nc": false}, {"id": "b-amanita-ovoidea", "lat": "Amanita ovoidea", "hr": "Amanita ovoidea", "shape": "convex", "nc": false}, {"id": "b-amanita-echinocephala", "lat": "Amanita echinocephala", "hr": "Ježasta muhara", "shape": "convex", "nc": false}, {"id": "b-amanita-aspera", "lat": "Amanita aspera", "hr": "Hrapava muhara kisobranasta muhara", "shape": "convex", "nc": false}, {"id": "b-amanita-inaurata", "lat": "Amanita inaurata", "hr": "Tigrasta preslica", "shape": "convex", "nc": false}, {"id": "b-amanita-livido", "lat": "Amanita livido", "hr": "Blijeda ovojnjača", "shape": "convex", "nc": false}, {"id": "b-amanita-umbrinolutea", "lat": "Amanita umbrinolutea", "hr": "Amanita umbrinolutea", "shape": "convex", "nc": false}, {"id": "b-amanita-crocea", "lat": "Amanita crocea", "hr": "Amanita crocea", "shape": "convex", "nc": false}, {"id": "b-volvariella-surrecta", "lat": "Volvariella surrecta", "hr": "Parazitska tobolčarka", "shape": "convex", "nc": false}, {"id": "b-lepiota-ventriosospora", "lat": "Lepiota ventriosospora", "hr": "Žutonoga štitarka", "shape": "umbrella-s", "nc": false}, {"id": "b-lepiota-alba", "lat": "Lepiota alba", "hr": "Bijela štitarka", "shape": "umbrella-s", "nc": false}, {"id": "b-lepiota-naucina", "lat": "Lepiota naucina", "hr": "Turkinja", "shape": "umbrella-s", "nc": false}, {"id": "b-leucoagaricus-holosericeus", "lat": "Leucoagaricus holosericeus", "hr": "Svilenasta pečurkovica", "shape": "umbrella-s", "nc": false}, {"id": "b-leucocoprinus-badhamii", "lat": "Leucocoprinus badhamii", "hr": "Leucocoprinus badhamii", "shape": "convex", "nc": false}, {"id": "b-cystoderma-amianthinum", "lat": "Cystoderma amianthinum", "hr": "Smeđa hrapavica", "shape": "small", "nc": false}, {"id": "b-chamaemyces-fracidus", "lat": "Chamaemyces fracidus", "hr": "Chamaemyces fracidus", "shape": "convex", "nc": false}, {"id": "b-limacella-illinita", "lat": "Limacella illinita", "hr": "Slinava ljepljivica", "shape": "convex", "nc": false}, {"id": "b-limacella-guttata", "lat": "Limacella guttata", "hr": "Kapljičasta ljepljivica", "shape": "convex", "nc": false}, {"id": "b-catathelasma-imperiale", "lat": "Catathelasma imperiale", "hr": "Golema dvoprstenka", "shape": "convex", "nc": false}, {"id": "b-coprinus-truncorum", "lat": "Coprinus truncorum", "hr": "Panjevkasta gnojištarka", "shape": "tall", "nc": false}, {"id": "b-coprinus-hemerobius", "lat": "Coprinus hemerobius", "hr": "Jednodnevna gnojištarka", "shape": "tall", "nc": false}, {"id": "b-coprinus-cinereus", "lat": "Coprinus cinereus", "hr": "Gljiva đubretara", "shape": "tall", "nc": false}, {"id": "b-agaricus-macrosporus", "lat": "Agaricus macrosporus", "hr": "Agaricus macrosporus", "shape": "convex", "nc": false}, {"id": "b-agaricus-bisporus", "lat": "Agaricus bisporus", "hr": "Plemenita pečurka", "shape": "convex", "nc": false}, {"id": "b-agaricus-arvensis", "lat": "Agaricus arvensis", "hr": "Ovčarska pečurka", "shape": "convex", "nc": false}, {"id": "b-agaricus-abruptibulbus", "lat": "Agaricus abruptibulbus", "hr": "Agaricus abruptibulbus", "shape": "convex", "nc": false}, {"id": "b-agaricus-perrarus", "lat": "Agaricus perrarus", "hr": "Vilovnjača", "shape": "convex", "nc": false}, {"id": "b-agaricus-benesii", "lat": "Agaricus benesii", "hr": "Benesijeva pečurka", "shape": "convex", "nc": false}, {"id": "b-agaricus-langei", "lat": "Agaricus langei", "hr": "Langeova pečurka", "shape": "convex", "nc": false}, {"id": "b-agaricus-vaporarius", "lat": "Agaricus vaporarius", "hr": "Resasta pečurka", "shape": "convex", "nc": false}, {"id": "b-agaricus-radicatus", "lat": "Agaricus radicatus", "hr": "Korijenasta pečurka", "shape": "convex", "nc": false}, {"id": "b-agaricus-purpurascens", "lat": "Agaricus purpurascens", "hr": "Purpurna pečurka", "shape": "convex", "nc": false}, {"id": "b-panaeolus-sphinctrinus", "lat": "Panaeolus sphinctrinus", "hr": "Zvonoliki smetištar", "shape": "conical", "nc": false}, {"id": "b-panaeolus-rickenii", "lat": "Panaeolus rickenii", "hr": "Rickenov smetištar", "shape": "conical", "nc": false}, {"id": "b-anellaria-semiovata", "lat": "Anellaria semiovata", "hr": "Prstenasti smetištar", "shape": "conical", "nc": false}, {"id": "b-psathyrella-candolleana", "lat": "Psathyrella candolleana", "hr": "Candolleova slabunjavka", "shape": "conical", "nc": false}, {"id": "b-psathyrella-prona", "lat": "Psathyrella prona", "hr": "Krivonoga slabunjavka", "shape": "conical", "nc": false}, {"id": "b-psathyrella-spadiceo-grisea", "lat": "Psathyrella spadiceo-grisea", "hr": "Smeđe-siva slabunjavka", "shape": "conical", "nc": false}, {"id": "b-psathyrella-velutina", "lat": "Psathyrella velutina", "hr": "Suzna cunjavka", "shape": "conical", "nc": false}, {"id": "b-stropharia-squamosa", "lat": "Stropharia squamosa", "hr": "Čehasta vitičarka", "shape": "convex", "nc": false}, {"id": "b-stropharia-eximia", "lat": "Stropharia eximia", "hr": "Golema vitičarka", "shape": "convex", "nc": false}, {"id": "b-agrocybe-dura", "lat": "Agrocybe dura", "hr": "Tvrda livadarka", "shape": "convex", "nc": false}, {"id": "b-agrocybe-erebia", "lat": "Agrocybe erebia", "hr": "Tamna livadarka", "shape": "convex", "nc": false}, {"id": "b-agrocybe-paludosa", "lat": "Agrocybe paludosa", "hr": "Močvarna livadarka", "shape": "convex", "nc": false}, {"id": "b-pholiota-squarroso-adiposa", "lat": "Pholiota squarroso-adiposa", "hr": "Masno-čehava ljuskavica", "shape": "convex", "nc": false}, {"id": "b-pholiota-aurivella", "lat": "Pholiota aurivella", "hr": "Pholiota aurivella", "shape": "convex", "nc": false}, {"id": "b-pholiota-destruens", "lat": "Pholiota destruens", "hr": "Razorna čehavka", "shape": "convex", "nc": false}, {"id": "b-pholiota-flammans", "lat": "Pholiota flammans", "hr": "Plamena ljuskavica", "shape": "convex", "nc": false}, {"id": "b-pholiota-carbonaria", "lat": "Pholiota carbonaria", "hr": "Ugljenarska ljuskavica", "shape": "convex", "nc": false}, {"id": "b-pholiota-flavida", "lat": "Pholiota flavida", "hr": "Žuta ljuskavica", "shape": "convex", "nc": false}, {"id": "b-pholiota-amara", "lat": "Pholiota amara", "hr": "Gorka ljuskavica", "shape": "convex", "nc": false}, {"id": "b-pholiotina-blattaria", "lat": "Pholiotina blattaria", "hr": "Okrunjena ljuskavica", "shape": "conical", "nc": false}, {"id": "b-bolbitius-vitellinus", "lat": "Bolbitius vitellinus", "hr": "Zlatan smetištar", "shape": "conical", "nc": false}, {"id": "b-gymnopilus-spectabilis", "lat": "Gymnopilus spectabilis", "hr": "Divna čehavka", "shape": "convex", "nc": false}, {"id": "b-hebeloma-radicosum", "lat": "Hebeloma radicosum", "hr": "Hebeloma radicosum", "shape": "convex", "nc": false}, {"id": "b-hebeloma-crustuliniforme", "lat": "Hebeloma crustuliniforme", "hr": "Hebeloma crustuliniforme", "shape": "convex", "nc": false}, {"id": "b-cortinarius-traganus", "lat": "Cortinarius traganus", "hr": "Cortinarius traganus", "shape": "convex", "nc": false}, {"id": "b-cortinarius-coerulescens", "lat": "Cortinarius coerulescens", "hr": "Modra koprenka", "shape": "convex", "nc": false}, {"id": "b-cortinarius-nebularis", "lat": "Cortinarius nebularis", "hr": "Maglasta koprenka", "shape": "convex", "nc": false}, {"id": "b-cortinarius-zinziberatus", "lat": "Cortinarius zinziberatus", "hr": "Đumbirova koprenka", "shape": "convex", "nc": false}, {"id": "b-cortinarius-collinitus", "lat": "Cortinarius collinitus", "hr": "Cortinarius collinitus", "shape": "convex", "nc": false}, {"id": "b-cortinarius-violaceus", "lat": "Cortinarius violaceus", "hr": "Ljubičasta koprenka", "shape": "convex", "nc": false}, {"id": "b-cortinarius-arcuatorum", "lat": "Cortinarius arcuatorum", "hr": "Svedena koprenka", "shape": "convex", "nc": false}, {"id": "b-cortinarius-glaucopus", "lat": "Cortinarius glaucopus", "hr": "Zelenkasta koprenka", "shape": "convex", "nc": false}, {"id": "b-cortinarius-bulliardi", "lat": "Cortinarius bulliardi", "hr": "Bulliardova koprenka", "shape": "convex", "nc": false}, {"id": "b-cortinarius-hinnuleus", "lat": "Cortinarius hinnuleus", "hr": "Jelenova koprenka", "shape": "convex", "nc": false}, {"id": "b-cortinarius-pholideus", "lat": "Cortinarius pholideus", "hr": "Cortinarius pholideus", "shape": "convex", "nc": false}, {"id": "b-cortinarius-nemorensis", "lat": "Cortinarius nemorensis", "hr": "Šumovita koprenka", "shape": "convex", "nc": false}, {"id": "b-dermocybe-cinnamomea", "lat": "Dermocybe cinnamomea", "hr": "Cimetasta crvenjača", "shape": "convex", "nc": false}, {"id": "b-dermocybe-anthracina", "lat": "Dermocybe anthracina", "hr": "Antracitna koprenka", "shape": "convex", "nc": false}, {"id": "b-dermocybe-cinnabarina", "lat": "Dermocybe cinnabarina", "hr": "Cinoberasta koprenka", "shape": "convex", "nc": false}, {"id": "b-inocybe-maculata", "lat": "Inocybe maculata", "hr": "Pjegava cjepača", "shape": "conical", "nc": false}, {"id": "b-inocybe-grammata", "lat": "Inocybe grammata", "hr": "Gravirana cjepača", "shape": "conical", "nc": false}, {"id": "b-inocybe-umbratica", "lat": "Inocybe umbratica", "hr": "Sjenovita cjepača", "shape": "conical", "nc": false}, {"id": "b-inocybe-sambucina", "lat": "Inocybe sambucina", "hr": "Bazgova cjepača", "shape": "conical", "nc": false}, {"id": "b-inocybe-godeyi", "lat": "Inocybe godeyi", "hr": "Trinijeva cjepača", "shape": "conical", "nc": false}, {"id": "b-inocybe-cincinnata", "lat": "Inocybe cincinnata", "hr": "Kovrčasta cjepača", "shape": "conical", "nc": false}, {"id": "b-inocybe-hirsuta", "lat": "Inocybe hirsuta", "hr": "Kuštrava cjepača", "shape": "conical", "nc": false}, {"id": "b-entoloma-aprile", "lat": "Entoloma aprile", "hr": "Travanjska rudoliska", "shape": "convex", "nc": false}, {"id": "b-entoloma-nitidum", "lat": "Entoloma nitidum", "hr": "Blistava rudoliska", "shape": "convex", "nc": false}, {"id": "b-entoloma-mougeoti", "lat": "Entoloma mougeoti", "hr": "Rumenkasta rudoliska", "shape": "convex", "nc": false}, {"id": "b-entoloma-mammosum", "lat": "Entoloma mammosum", "hr": "Bradavičava rudoliska", "shape": "convex", "nc": false}, {"id": "b-entoloma-staurosporum", "lat": "Entoloma staurosporum", "hr": "Križastotrusna rudoliska", "shape": "convex", "nc": false}, {"id": "b-clitopilus-cretatus", "lat": "Clitopilus cretatus", "hr": "Kredasta brašnjača", "shape": "shelf", "nc": false}, {"id": "b-rhodocybe-truncata", "lat": "Rhodocybe truncata", "hr": "Mirisava vitezovka", "shape": "convex", "nc": false}, {"id": "b-pluteus-atricapillus", "lat": "Pluteus atricapillus", "hr": "Pluteus atricapillus", "shape": "convex", "nc": false}, {"id": "b-pluteus-leoninus", "lat": "Pluteus leoninus", "hr": "Lavlja krovnjača", "shape": "convex", "nc": false}, {"id": "b-collybia-dryophila", "lat": "Collybia dryophila", "hr": "Collybia dryophila", "shape": "flat", "nc": false}, {"id": "b-collybia-platyphylla", "lat": "Collybia platyphylla", "hr": "Collybia platyphylla", "shape": "flat", "nc": false}, {"id": "b-collybia-exculpta", "lat": "Collybia exculpta", "hr": "Padobranasta plosnatica", "shape": "flat", "nc": false}, {"id": "b-collybia-acervata", "lat": "Collybia acervata", "hr": "Zadružna plosnatica", "shape": "flat", "nc": false}, {"id": "b-collybia-peronata", "lat": "Collybia peronata", "hr": "Ljuta plosnatica", "shape": "flat", "nc": false}, {"id": "b-collybia-hariolorum", "lat": "Collybia hariolorum", "hr": "Vračarska plosnatica", "shape": "flat", "nc": false}, {"id": "b-oudemansiella-longipes", "lat": "Oudemansiella longipes", "hr": "Dugonoga korjenjača", "shape": "convex", "nc": false}, {"id": "b-strobilurus-stephanocystis", "lat": "Strobilurus stephanocystis", "hr": "Tanašna češerka", "shape": "small", "nc": false}, {"id": "b-strobilurus-tenacellus", "lat": "Strobilurus tenacellus", "hr": "Gorka češerka", "shape": "small", "nc": false}, {"id": "b-marasmius-scorodonius", "lat": "Marasmius scorodonius", "hr": "Marasmius scorodonius", "shape": "small", "nc": false}, {"id": "b-marasmius-wynnei", "lat": "Marasmius wynnei", "hr": "Marasmius wynnei", "shape": "small", "nc": false}, {"id": "b-marasmius-bulliardii", "lat": "Marasmius bulliardii", "hr": "Bulliardova čamavka", "shape": "small", "nc": false}, {"id": "b-marasmius-androsaceus", "lat": "Marasmius androsaceus", "hr": "Nitasta čamavka", "shape": "small", "nc": false}, {"id": "b-marasmiellus-ramealis", "lat": "Marasmiellus ramealis", "hr": "Bijela sušičarka", "shape": "convex", "nc": false}, {"id": "b-micromphale-brassicolens", "lat": "Micromphale brassicolens", "hr": "Kupusna čamavka", "shape": "coral", "nc": false}, {"id": "b-micromphale-foetidum", "lat": "Micromphale foetidum", "hr": "Smrdljiva čamavka", "shape": "convex", "nc": false}, {"id": "b-mycena-rosella", "lat": "Mycena rosella", "hr": "Crvena šljemovka", "shape": "small", "nc": false}, {"id": "b-mycena-callopoda", "lat": "Mycena callopoda", "hr": "Ljeponoga šljemovka", "shape": "small", "nc": false}, {"id": "b-mycena-flavipes", "lat": "Mycena flavipes", "hr": "Žutonoga šljemovka", "shape": "small", "nc": false}, {"id": "b-leucopaxillus-amarus", "lat": "Leucopaxillus amarus", "hr": "Gorka debeljača", "shape": "funnel", "nc": false}, {"id": "b-tricholoma-georgi", "lat": "Tricholoma georgi", "hr": "Žuta đurđevača", "shape": "convex", "nc": false}, {"id": "b-tricholoma-viridilutescens", "lat": "Tricholoma viridilutescens", "hr": "Limunasta vitezovka", "shape": "convex", "nc": false}, {"id": "b-tricholoma-flavovirens", "lat": "Tricholoma flavovirens", "hr": "Zelenkasta vitezovka", "shape": "convex", "nc": false}, {"id": "b-tricholoma-auratum", "lat": "Tricholoma auratum", "hr": "Zlatna zelenka", "shape": "convex", "nc": false}, {"id": "b-tricholoma-albobrunneum", "lat": "Tricholoma albobrunneum", "hr": "Tricholoma albobrunneum", "shape": "convex", "nc": false}, {"id": "b-tricholoma-batschii", "lat": "Tricholoma batschii", "hr": "Batschijeva vitezovka", "shape": "convex", "nc": false}, {"id": "b-tricholoma-bufonium", "lat": "Tricholoma bufonium", "hr": "Žablja vitezovka", "shape": "convex", "nc": false}, {"id": "b-tricholoma-imbricatum", "lat": "Tricholoma imbricatum", "hr": "Ljuskava vitezovka", "shape": "convex", "nc": false}, {"id": "b-tricholoma-vaccinum", "lat": "Tricholoma vaccinum", "hr": "Tricholoma vaccinum", "shape": "convex", "nc": false}, {"id": "b-tricholoma-virgatum", "lat": "Tricholoma virgatum", "hr": "Tricholoma virgatum", "shape": "convex", "nc": false}, {"id": "b-tricholoma-tigrinum", "lat": "Tricholoma tigrinum", "hr": "Tricholoma tigrinum", "shape": "convex", "nc": false}, {"id": "b-tricholoma-squarrulosum", "lat": "Tricholoma squarrulosum", "hr": "Sitnočehava vitezovka", "shape": "convex", "nc": false}, {"id": "b-tricholoma-atrosquamosum", "lat": "Tricholoma atrosquamosum", "hr": "Tamnočehasta vitezovka", "shape": "convex", "nc": false}, {"id": "b-tricholoma-orirubens", "lat": "Tricholoma orirubens", "hr": "Tricholoma orirubens", "shape": "convex", "nc": false}, {"id": "b-tricholoma-album", "lat": "Tricholoma album", "hr": "Bijela vitezovka", "shape": "convex", "nc": false}, {"id": "b-tricholoma-acerbum", "lat": "Tricholoma acerbum", "hr": "Gorka vitezovka", "shape": "convex", "nc": false}, {"id": "b-lepista-irina", "lat": "Lepista irina", "hr": "Perunikina kružoliska", "shape": "convex", "nc": false}, {"id": "b-lepista-caespitosa", "lat": "Lepista caespitosa", "hr": "Busenasta koturnica", "shape": "convex", "nc": false}, {"id": "b-lepista-rickenii", "lat": "Lepista rickenii", "hr": "Rickenova koturnica", "shape": "convex", "nc": false}, {"id": "b-melanoleuca-grammopodia", "lat": "Melanoleuca grammopodia", "hr": "Prugasta gustoliska", "shape": "flat", "nc": false}, {"id": "b-melanoleuca-subbrevipes", "lat": "Melanoleuca subbrevipes", "hr": "Kratkonoga gustoliska", "shape": "flat", "nc": false}, {"id": "b-melanoleuca-strictipes", "lat": "Melanoleuca strictipes", "hr": "Tankonoga gustoliska", "shape": "flat", "nc": false}, {"id": "b-melanoleuca-excissa", "lat": "Melanoleuca excissa", "hr": "Pepeljasta gustoliska", "shape": "flat", "nc": false}, {"id": "b-clitocybe-inversa", "lat": "Clitocybe inversa", "hr": "Clitocybe inversa", "shape": "funnel", "nc": false}, {"id": "b-clitocybe-gilva", "lat": "Clitocybe gilva", "hr": "Clitocybe gilva", "shape": "funnel", "nc": false}, {"id": "b-clitocybe-vermicularis", "lat": "Clitocybe vermicularis", "hr": "Ariševa grlašica", "shape": "funnel", "nc": false}, {"id": "b-clitocybe-maxima", "lat": "Clitocybe maxima", "hr": "Velika grlašica", "shape": "funnel", "nc": false}, {"id": "b-clitocybe-costata", "lat": "Clitocybe costata", "hr": "Resasta grlašica", "shape": "funnel", "nc": false}, {"id": "b-clitocybe-alexandri", "lat": "Clitocybe alexandri", "hr": "Aleksandrova grlašica", "shape": "funnel", "nc": false}, {"id": "b-clitocybe-inornata", "lat": "Clitocybe inornata", "hr": "Neukrašena grlašica", "shape": "funnel", "nc": false}, {"id": "b-clitocybe-cerussata", "lat": "Clitocybe cerussata", "hr": "Clitocybe cerussata", "shape": "funnel", "nc": false}, {"id": "b-lyophyllum-immundum", "lat": "Lyophyllum immundum", "hr": "Prljava vitezovka", "shape": "convex", "nc": false}, {"id": "b-lyophyllum-infumatum", "lat": "Lyophyllum infumatum", "hr": "Zadimljena vitezovka", "shape": "convex", "nc": false}, {"id": "b-lyophyllum-semitale", "lat": "Lyophyllum semitale", "hr": "Polukopljasta vitezovka", "shape": "convex", "nc": false}, {"id": "b-lactarius-porninsis", "lat": "Lactarius porninsis", "hr": "Ariševa mliječnica", "shape": "funnel", "nc": false}, {"id": "b-lactarius-cilicioides", "lat": "Lactarius cilicioides", "hr": "Ćilimasta mliječnica", "shape": "funnel", "nc": false}, {"id": "b-lactarius-scrobiculatus", "lat": "Lactarius scrobiculatus", "hr": "Jamičasta mliječnica", "shape": "funnel", "nc": false}, {"id": "b-lactarius-zonarius", "lat": "Lactarius zonarius", "hr": "Pojasasta mliječnica", "shape": "funnel", "nc": false}, {"id": "b-lactarius-luteolus", "lat": "Lactarius luteolus", "hr": "Žućkasta mliječnica", "shape": "funnel", "nc": false}, {"id": "b-lactarius-trivialis", "lat": "Lactarius trivialis", "hr": "Lactarius trivialis", "shape": "funnel", "nc": false}, {"id": "b-lactarius-contorversus", "lat": "Lactarius contorversus", "hr": "Topolova mliječnica", "shape": "funnel", "nc": false}, {"id": "b-lactarius-pergamenus", "lat": "Lactarius pergamenus", "hr": "Pergamenasta mliječnica", "shape": "funnel", "nc": false}, {"id": "b-lactarius-fuliginosus", "lat": "Lactarius fuliginosus", "hr": "Čađava mliječnica", "shape": "funnel", "nc": false}, {"id": "b-lactarius-picinus", "lat": "Lactarius picinus", "hr": "Smrekova mliječnica", "shape": "funnel", "nc": false}, {"id": "b-lactarius-uvidus", "lat": "Lactarius uvidus", "hr": "Lactarius uvidus", "shape": "funnel", "nc": false}, {"id": "b-russula-aeruginea", "lat": "Russula aeruginea", "hr": "Zelena krasnica", "shape": "flat", "nc": false}, {"id": "b-russula-parazurea", "lat": "Russula parazurea", "hr": "Modrikasta krasnica", "shape": "flat", "nc": false}, {"id": "b-russula-amoena", "lat": "Russula amoena", "hr": "Ljupka krasnica", "shape": "flat", "nc": false}, {"id": "b-russula-integra", "lat": "Russula integra", "hr": "Smeđokoža golubica", "shape": "flat", "nc": false}, {"id": "b-russula-mustelina", "lat": "Russula mustelina", "hr": "Russula mustelina", "shape": "flat", "nc": false}, {"id": "b-russula-vitellina", "lat": "Russula vitellina", "hr": "Žumanjčana krasnica", "shape": "flat", "nc": false}, {"id": "b-russula-fellea", "lat": "Russula fellea", "hr": "Žučna krasnica", "shape": "flat", "nc": false}, {"id": "b-russula-illota", "lat": "Russula illota", "hr": "Prljava krasnica", "shape": "flat", "nc": false}, {"id": "b-russula-laurocerasi", "lat": "Russula laurocerasi", "hr": "Bademova krasnica", "shape": "flat", "nc": false}, {"id": "b-russula-pseudodelica", "lat": "Russula pseudodelica", "hr": "Lažna modrolisna krasnica", "shape": "flat", "nc": false}, {"id": "b-russula-viscida", "lat": "Russula viscida", "hr": "Ljepljiva krasnica", "shape": "flat", "nc": false}, {"id": "b-russula-amethystina", "lat": "Russula amethystina", "hr": "Ametistna krasnica", "shape": "flat", "nc": false}, {"id": "b-russula-puellaris", "lat": "Russula puellaris", "hr": "Djevojačka krasnica", "shape": "flat", "nc": false}, {"id": "b-russula-versicolor", "lat": "Russula versicolor", "hr": "Šarolika krasnica", "shape": "flat", "nc": false}, {"id": "b-russula-acrifolia", "lat": "Russula acrifolia", "hr": "Ljutolisna krasnica", "shape": "flat", "nc": false}, {"id": "b-russula-pectinatoides", "lat": "Russula pectinatoides", "hr": "Češljolika krasnica", "shape": "flat", "nc": false}, {"id": "b-russula-amoenolens", "lat": "Russula amoenolens", "hr": "Camembert krasnica", "shape": "flat", "nc": false}, {"id": "b-russula-sanguinea", "lat": "Russula sanguinea", "hr": "Krvava krasnica", "shape": "flat", "nc": false}, {"id": "b-russula-badia", "lat": "Russula badia", "hr": "Cedrova krasnica", "shape": "flat", "nc": false}, {"id": "b-asterophora-lycoperdoides", "lat": "Asterophora lycoperdoides", "hr": "Obična dvotrusnica", "shape": "small", "nc": false}, {"id": "b-hygrophorus-pudorinus", "lat": "Hygrophorus pudorinus", "hr": "Hygrophorus pudorinus", "shape": "convex", "nc": false}, {"id": "b-hygrophorus-penarius", "lat": "Hygrophorus penarius", "hr": "Suha puževica", "shape": "convex", "nc": false}, {"id": "b-hygrophorus-leporinus", "lat": "Hygrophorus leporinus", "hr": "Zečja sočnica", "shape": "convex", "nc": false}, {"id": "b-hygrophorus-queletii", "lat": "Hygrophorus queletii", "hr": "Queletova puževica", "shape": "convex", "nc": false}, {"id": "b-hygrophorus-chrysodon", "lat": "Hygrophorus chrysodon", "hr": "Zlatnožuta puževica", "shape": "convex", "nc": false}, {"id": "b-hygrophorus-co", "lat": "Hygrophorus co", "hr": "Hygrophorus co", "shape": "convex", "nc": false}, {"id": "b-hygrophorus-gliocyclus", "lat": "Hygrophorus gliocyclus", "hr": "Sluzavo-prstenasta puževica", "shape": "convex", "nc": false}, {"id": "b-hygrophorus-dichrous", "lat": "Hygrophorus dichrous", "hr": "Grabova puževica", "shape": "convex", "nc": false}, {"id": "b-hygrophorus-agathosmus", "lat": "Hygrophorus agathosmus", "hr": "Mirisna puževica", "shape": "convex", "nc": false}, {"id": "b-hygrophorus-lucorum", "lat": "Hygrophorus lucorum", "hr": "Ariševa puževica", "shape": "convex", "nc": false}, {"id": "b-hygrophorus-hypotheius", "lat": "Hygrophorus hypotheius", "hr": "Hygrophorus hypotheius", "shape": "convex", "nc": false}, {"id": "b-hygrophorus-leucophaeus", "lat": "Hygrophorus leucophaeus", "hr": "Dvobojna puževica", "shape": "convex", "nc": false}, {"id": "b-hygrophorus-erubescens", "lat": "Hygrophorus erubescens", "hr": "Crvenkasta puževica", "shape": "convex", "nc": false}, {"id": "b-hygrophorus-discoideus", "lat": "Hygrophorus discoideus", "hr": "Plosnata puževica", "shape": "convex", "nc": false}, {"id": "b-hygrophorus-aureus", "lat": "Hygrophorus aureus", "hr": "Zlatna puževica", "shape": "convex", "nc": false}, {"id": "b-hygrocybe-konradii", "lat": "Hygrocybe konradii", "hr": "Hygrocybe konradii", "shape": "conical", "nc": false}, {"id": "b-hygrocybe-obrussea", "lat": "Hygrocybe obrussea", "hr": "Rascijepljena vlažnica", "shape": "conical", "nc": false}, {"id": "b-hygrocybe-brevispora", "lat": "Hygrocybe brevispora", "hr": "Kratkotrusna vlažnica", "shape": "conical", "nc": false}, {"id": "b-hygrocybe-citrina", "lat": "Hygrocybe citrina", "hr": "Limunasta vlažnica", "shape": "conical", "nc": false}, {"id": "b-hygrocybe-quieta", "lat": "Hygrocybe quieta", "hr": "Spavajuća vlažnica", "shape": "conical", "nc": false}, {"id": "b-hygrocybe-intermedius", "lat": "Hygrocybe intermedius", "hr": "Srednja vlažnica", "shape": "conical", "nc": false}, {"id": "b-hygrocybe-coccineus", "lat": "Hygrocybe coccineus", "hr": "Hygrocybe coccineus", "shape": "conical", "nc": false}, {"id": "b-hygrocybe-nigrescens", "lat": "Hygrocybe nigrescens", "hr": "Hygrocybe nigrescens", "shape": "conical", "nc": false}, {"id": "b-hygrocybe-laeta", "lat": "Hygrocybe laeta", "hr": "Klizava vlažnica", "shape": "conical", "nc": false}, {"id": "b-hygrocybe-reai", "lat": "Hygrocybe reai", "hr": "Gorka vlažnica", "shape": "conical", "nc": false}, {"id": "b-hygrocybe-ovina", "lat": "Hygrocybe ovina", "hr": "Maslinasto-crna vlažnica", "shape": "conical", "nc": false}, {"id": "b-camarophyllus-niveus", "lat": "Camarophyllus niveus", "hr": "Snježna sočnica", "shape": "convex", "nc": false}, {"id": "b-camarophyllus-russocoriaceus", "lat": "Camarophyllus russocoriaceus", "hr": "Mirisna sočnica", "shape": "convex", "nc": false}, {"id": "b-cantharellus-friesii", "lat": "Cantharellus friesii", "hr": "Sitna lisičica", "shape": "funnel", "nc": false}, {"id": "b-cantharellus-amethysteus", "lat": "Cantharellus amethysteus", "hr": "Ametistna lisičica", "shape": "funnel", "nc": false}, {"id": "b-cantharellus-subcibarius", "lat": "Cantharellus subcibarius", "hr": "Blijeda lisičica", "shape": "funnel", "nc": false}, {"id": "b-gomphus-clavatus", "lat": "Gomphus clavatus", "hr": "Grimizna lisičica", "shape": "funnel", "nc": false}, {"id": "b-cantharellus-cinerus", "lat": "Cantharellus cinerus", "hr": "Pepeljasta lisičarka", "shape": "funnel", "nc": false}, {"id": "b-xeromphalina-campanella", "lat": "Xeromphalina campanella", "hr": "Združena zvonca", "shape": "convex", "nc": false}, {"id": "b-gomphidius-maculatus", "lat": "Gomphidius maculatus", "hr": "Pjegavi slinar", "shape": "convex", "nc": false}, {"id": "b-chroogomphus-rutilus", "lat": "Chroogomphus rutilus", "hr": "Borov slinavac", "shape": "convex", "nc": false}, {"id": "b-chroogomphus-helveticus", "lat": "Chroogomphus helveticus", "hr": "Švicarski prljavac", "shape": "convex", "nc": false}, {"id": "b-pleurotus-columbinus", "lat": "Pleurotus columbinus", "hr": "Golubinja krivonoška", "shape": "shelf", "nc": false}, {"id": "b-pleurotus-sapidus", "lat": "Pleurotus sapidus", "hr": "Pleurotus sapidus", "shape": "shelf", "nc": false}, {"id": "b-pleurocybella-porrigens", "lat": "Pleurocybella porrigens", "hr": "Bjelouška", "shape": "convex", "nc": false}, {"id": "b-panus-tigrinus", "lat": "Panus tigrinus", "hr": "Panus tigrinus", "shape": "funnel", "nc": false}, {"id": "b-crepidotus-variabilis", "lat": "Crepidotus variabilis", "hr": "Promjenljiva batrljica", "shape": "shelf", "nc": false}, {"id": "b-phylloporus-rhodoxanthus", "lat": "Phylloporus rhodoxanthus", "hr": "Phylloporus rhodoxanthus", "shape": "convex", "nc": false}, {"id": "b-boletus-fechtneri", "lat": "Boletus fechtneri", "hr": "Fechtnerov vrganj", "shape": "convex", "nc": false}, {"id": "b-boletus-impolitus", "lat": "Boletus impolitus", "hr": "Prosti vrganj", "shape": "convex", "nc": false}, {"id": "b-boletus-fragrans", "lat": "Boletus fragrans", "hr": "Mirisavi vrganj", "shape": "convex", "nc": false}, {"id": "b-boletus-junquilleus", "lat": "Boletus junquilleus", "hr": "Žuta glatkonoška", "shape": "convex", "nc": false}, {"id": "b-boletus-albidus", "lat": "Boletus albidus", "hr": "Gorki vrganj", "shape": "convex", "nc": false}, {"id": "b-boletus-satanoides", "lat": "Boletus satanoides", "hr": "Vragoljasti vrganj", "shape": "convex", "nc": false}, {"id": "b-boletus-rhodoxanthus", "lat": "Boletus rhodoxanthus", "hr": "Purpurno-žuti vrganj", "shape": "convex", "nc": false}, {"id": "b-boletus-purpureus", "lat": "Boletus purpureus", "hr": "Skerletni vrganj", "shape": "convex", "nc": false}, {"id": "b-boletus-dupainii", "lat": "Boletus dupainii", "hr": "Dupainov vrganj", "shape": "convex", "nc": false}, {"id": "b-xerocomus-spadiceus", "lat": "Xerocomus spadiceus", "hr": "Smeđi baršunovac", "shape": "convex", "nc": false}, {"id": "b-xerocomus-parasticus", "lat": "Xerocomus parasticus", "hr": "Parazitski baršunovac", "shape": "convex", "nc": false}, {"id": "b-leccinum-testaceo-scabrum", "lat": "Leccinum testaceo-scabrum", "hr": "Opekasti djed", "shape": "convex", "nc": false}, {"id": "b-leccinum-criseum", "lat": "Leccinum criseum", "hr": "Leccinum criseum", "shape": "convex", "nc": false}, {"id": "b-leccinum-holopus", "lat": "Leccinum holopus", "hr": "Bijeli brezov djed", "shape": "convex", "nc": false}, {"id": "b-leccinum-duriusculum", "lat": "Leccinum duriusculum", "hr": "Topolov djed", "shape": "convex", "nc": false}, {"id": "b-suillus-aeruginascens", "lat": "Suillus aeruginascens", "hr": "Suillus aeruginascens", "shape": "convex", "nc": false}, {"id": "b-suillus-placidus", "lat": "Suillus placidus", "hr": "Ljupki vrganj", "shape": "convex", "nc": false}, {"id": "b-gyrodon-lividus", "lat": "Gyrodon lividus", "hr": "Žuti johovac", "shape": "convex", "nc": false}, {"id": "b-polyporus-confluens", "lat": "Polyporus confluens", "hr": "Žemljača", "shape": "convex", "nc": false}, {"id": "b-polyporus-ovinus", "lat": "Polyporus ovinus", "hr": "Ovčje vime", "shape": "convex", "nc": false}, {"id": "b-polyporus-cristatus", "lat": "Polyporus cristatus", "hr": "Zeleno vime", "shape": "convex", "nc": false}, {"id": "b-polyporus-pes-caprae", "lat": "Polyporus pes-caprae", "hr": "Polyporus pes-caprae", "shape": "convex", "nc": false}, {"id": "b-polyporus-giganteus", "lat": "Polyporus giganteus", "hr": "Orijaška zrašćenka", "shape": "convex", "nc": false}, {"id": "b-polyporus-montanus", "lat": "Polyporus montanus", "hr": "Planinski žbunac", "shape": "convex", "nc": false}, {"id": "b-polyporus-frandosus", "lat": "Polyporus frandosus", "hr": "Polyporus frandosus", "shape": "convex", "nc": false}, {"id": "b-polyporus-intybaceus", "lat": "Polyporus intybaceus", "hr": "Polyporus intybaceus", "shape": "convex", "nc": false}, {"id": "b-polyporus-squamosus", "lat": "Polyporus squamosus", "hr": "Polyporus squamosus", "shape": "convex", "nc": false}, {"id": "b-polyporus-lentus", "lat": "Polyporus lentus", "hr": "Polyporus lentus", "shape": "convex", "nc": false}, {"id": "b-polyporus-sulphureus", "lat": "Polyporus sulphureus", "hr": "Polyporus sulphureus", "shape": "convex", "nc": false}, {"id": "b-polyporus-lepideus", "lat": "Polyporus lepideus", "hr": "Svibanjski rupičar", "shape": "convex", "nc": false}, {"id": "b-polyporus-mori", "lat": "Polyporus mori", "hr": "Saćasti rupičar", "shape": "convex", "nc": false}, {"id": "b-polyporus-arcularius", "lat": "Polyporus arcularius", "hr": "Uglati rupičar", "shape": "convex", "nc": false}, {"id": "b-polyporus-badius", "lat": "Polyporus badius", "hr": "Crnonogi rupičar", "shape": "convex", "nc": false}, {"id": "b-merulius-tremellosus", "lat": "Merulius tremellosus", "hr": "Naborana hladetina", "shape": "coral", "nc": false}, {"id": "b-inonotus-dryadeus", "lat": "Inonotus dryadeus", "hr": "Hrastova guba", "shape": "shelf", "nc": false}, {"id": "b-osmoporus-odoratus", "lat": "Osmoporus odoratus", "hr": "Mirisavi anisovac", "shape": "convex", "nc": false}, {"id": "b-daedalea-quercina", "lat": "Daedalea quercina", "hr": "Hrastova raznocijevka", "shape": "shelf", "nc": false}, {"id": "b-daedalea-confragosa", "lat": "Daedalea confragosa", "hr": "Hrapava raznocijevka", "shape": "shelf", "nc": false}, {"id": "b-thelephora-palmata", "lat": "Thelephora palmata", "hr": "Smrdljiva palma", "shape": "coral", "nc": false}, {"id": "b-sarcodon-badium", "lat": "Sarcodon badium", "hr": "Smeđasta ježevica", "shape": "convex", "nc": false}, {"id": "b-hydnum-rufescens", "lat": "Hydnum rufescens", "hr": "Ježek", "shape": "convex", "nc": false}, {"id": "b-calodon-aurantiacum", "lat": "Calodon aurantiacum", "hr": "Narančasta plutovka", "shape": "coral", "nc": false}, {"id": "b-calodon-compactum", "lat": "Calodon compactum", "hr": "Zbijena plutovka", "shape": "coral", "nc": false}, {"id": "b-dryodon-erinaceus", "lat": "Dryodon erinaceus", "hr": "Resasti igličar", "shape": "coral", "nc": false}, {"id": "b-pleurodon-auriscalpium", "lat": "Pleurodon auriscalpium", "hr": "Obična žličarka", "shape": "coral", "nc": false}, {"id": "b-sparassis-laminosa", "lat": "Sparassis laminosa", "hr": "Sparassis laminosa", "shape": "coral", "nc": false}, {"id": "b-clavaria-flava", "lat": "Clavaria flava", "hr": "Clavaria flava", "shape": "coral", "nc": false}, {"id": "b-clavaria-aurea", "lat": "Clavaria aurea", "hr": "Clavaria aurea", "shape": "coral", "nc": false}, {"id": "b-clavaria-rufescens", "lat": "Clavaria rufescens", "hr": "Clavaria rufescens", "shape": "coral", "nc": false}, {"id": "b-clavaria-formosa", "lat": "Clavaria formosa", "hr": "Clavaria formosa", "shape": "coral", "nc": false}, {"id": "b-clavaria-pallida", "lat": "Clavaria pallida", "hr": "Clavaria pallida", "shape": "coral", "nc": false}, {"id": "b-clavaria-lavendula", "lat": "Clavaria lavendula", "hr": "Ljubičasta capica", "shape": "coral", "nc": false}, {"id": "b-clavaria-cristata", "lat": "Clavaria cristata", "hr": "Grebenasta griva", "shape": "coral", "nc": false}, {"id": "b-clavaria-rugosa", "lat": "Clavaria rugosa", "hr": "Clavaria rugosa", "shape": "coral", "nc": false}, {"id": "b-clavaria-fennica", "lat": "Clavaria fennica", "hr": "Žuto-čađava capica", "shape": "coral", "nc": false}, {"id": "b-clavaria-invalii", "lat": "Clavaria invalii", "hr": "Crnogorična capica", "shape": "coral", "nc": false}, {"id": "b-clavaria-stricta", "lat": "Clavaria stricta", "hr": "Uska griva", "shape": "coral", "nc": false}, {"id": "b-clavaria-vermicularis", "lat": "Clavaria vermicularis", "hr": "Crvolika kijačica", "shape": "coral", "nc": false}, {"id": "b-pterula-multifida", "lat": "Pterula multifida", "hr": "Metlica", "shape": "coral", "nc": false}, {"id": "b-calocera-viscosa", "lat": "Calocera viscosa", "hr": "Calocera viscosa", "shape": "coral", "nc": false}, {"id": "b-clavaria-pistillaris", "lat": "Clavaria pistillaris", "hr": "Clavaria pistillaris", "shape": "coral", "nc": false}, {"id": "b-clavaria-truncata", "lat": "Clavaria truncata", "hr": "Clavaria truncata", "shape": "coral", "nc": false}, {"id": "b-tremellodon-gelatinosum", "lat": "Tremellodon gelatinosum", "hr": "Tremellodon gelatinosum", "shape": "coral", "nc": false}, {"id": "b-tremella-mesenterica", "lat": "Tremella mesenterica", "hr": "Zlatnožuta drhtalica", "shape": "ear", "nc": false}, {"id": "b-exidia-galndulosa", "lat": "Exidia galndulosa", "hr": "Bradavičava žljezdarica", "shape": "ear", "nc": false}, {"id": "b-tremiscus-helvelloides", "lat": "Tremiscus helvelloides", "hr": "Tremiscus helvelloides", "shape": "coral", "nc": false}, {"id": "b-lycoperdon-caelatum", "lat": "Lycoperdon caelatum", "hr": "Lycoperdon caelatum", "shape": "ball", "nc": false}, {"id": "b-lycoperdon-mammaeforme", "lat": "Lycoperdon mammaeforme", "hr": "Sisata puhara", "shape": "ball", "nc": false}, {"id": "b-lycoperdon-saccatum", "lat": "Lycoperdon saccatum", "hr": "Lycoperdon saccatum", "shape": "ball", "nc": false}, {"id": "b-lycoperdon-umbrinum", "lat": "Lycoperdon umbrinum", "hr": "Dlakava puhara", "shape": "ball", "nc": false}, {"id": "b-bovista-plumbea", "lat": "Bovista plumbea", "hr": "Olovasta jajača", "shape": "ball", "nc": false}, {"id": "b-rhizopogon-roseolus", "lat": "Rhizopogon roseolus", "hr": "Ružičasta opletenka", "shape": "coral", "nc": false}, {"id": "b-scleroderma-geaster", "lat": "Scleroderma geaster", "hr": "Krumpirača", "shape": "ball", "nc": false}, {"id": "b-geastrum-triplex", "lat": "Geastrum triplex", "hr": "Troslojna zvjezdača", "shape": "ball", "nc": false}, {"id": "b-geastrum-fimbriatum", "lat": "Geastrum fimbriatum", "hr": "Geastrum fimbriatum", "shape": "ball", "nc": false}, {"id": "b-geastrum-pectinatum", "lat": "Geastrum pectinatum", "hr": "Grebenasta zvjezdača", "shape": "ball", "nc": false}, {"id": "b-astraeus-hygrometricus", "lat": "Astraeus hygrometricus", "hr": "Upojna zvjezdača", "shape": "coral", "nc": false}, {"id": "b-clathrus-cancellatus", "lat": "Clathrus cancellatus", "hr": "Vještičino srce", "shape": "ball", "nc": false}, {"id": "b-phallus-hadriani", "lat": "Phallus hadriani", "hr": "Hadrianov stršak", "shape": "phallus", "nc": false}, {"id": "b-sebacina-laciniata", "lat": "Sebacina laciniata", "hr": "Odrpanka", "shape": "coral", "nc": false}, {"id": "b-spathularia-flavida", "lat": "Spathularia flavida", "hr": "Žuta lopatica", "shape": "coral", "nc": false}, {"id": "b-bulgaria-inquinans", "lat": "Bulgaria inquinans", "hr": "Bugarkinja", "shape": "coral", "nc": false}, {"id": "b-lycogala-epidendron", "lat": "Lycogala epidendron", "hr": "Krvotočna puhara", "shape": "coral", "nc": false}, {"id": "b-arcyria-denudata", "lat": "Arcyria denudata", "hr": "Ogoljela metlica", "shape": "coral", "nc": false}, {"id": "b-tubifera-ferruginosa", "lat": "Tubifera ferruginosa", "hr": "Crvena sluznjača", "shape": "coral", "nc": false}, {"id": "b-fuligo-septica", "lat": "Fuligo septica", "hr": "Žuta sluznjača", "shape": "coral", "nc": false}, {"id": "b-xylaria-polymorpha", "lat": "Xylaria polymorpha", "hr": "Xylaria polymorpha", "shape": "coral", "nc": false}, {"id": "b-xylaria-hypoxylon", "lat": "Xylaria hypoxylon", "hr": "Rogata drvarica", "shape": "coral", "nc": false}, {"id": "b-morchella-deliciosa", "lat": "Morchella deliciosa", "hr": "Divan smrčak", "shape": "tall", "nc": false}, {"id": "b-morchella-pragensis", "lat": "Morchella pragensis", "hr": "Praški smrčak", "shape": "tall", "nc": false}, {"id": "b-mitrophora-hybrida", "lat": "Mitrophora hybrida", "hr": "Mitrophora hybrida", "shape": "coral", "nc": false}, {"id": "b-verpa-bohemica", "lat": "Verpa bohemica", "hr": "Češka smrčkovica", "shape": "tall", "nc": false}, {"id": "b-verpa-digitaliformis", "lat": "Verpa digitaliformis", "hr": "Prstasta smrčkovica", "shape": "tall", "nc": false}, {"id": "b-helvella-crispa", "lat": "Helvella crispa", "hr": "Jesenski hrčak", "shape": "tall", "nc": false}, {"id": "b-helvella-lacunosa", "lat": "Helvella lacunosa", "hr": "Helvella lacunosa", "shape": "tall", "nc": false}, {"id": "b-helvella-elastica", "lat": "Helvella elastica", "hr": "Elastični hrčak", "shape": "tall", "nc": false}, {"id": "b-gyromitra-fastigiata", "lat": "Gyromitra fastigiata", "hr": "Rogljasti hrčak", "shape": "tall", "nc": false}, {"id": "b-tuber-brumale", "lat": "Tuber brumale", "hr": "Tuber brumale", "shape": "ball", "nc": false}, {"id": "b-tuber-melanosporum", "lat": "Tuber melanosporum", "hr": "Tuber melanosporum", "shape": "ball", "nc": false}, {"id": "b-elaphomyces-granulatus", "lat": "Elaphomyces granulatus", "hr": "Elaphomyces granulatus", "shape": "coral", "nc": false}, {"id": "b-choiromyces-maeandriformis", "lat": "Choiromyces maeandriformis", "hr": "Vijugavo-mesnata mirisnica", "shape": "coral", "nc": false}, {"id": "b-otidea-onotica", "lat": "Otidea onotica", "hr": "Otidea onotica", "shape": "ear", "nc": false}, {"id": "b-caloscypha-fulgens", "lat": "Caloscypha fulgens", "hr": "Sjajni pehar", "shape": "coral", "nc": false}, {"id": "b-aleuria-aurantia", "lat": "Aleuria aurantia", "hr": "Narančasto-crvena zdjeličarka", "shape": "funnel", "nc": false}, {"id": "b-sarcoscypha-coccinea", "lat": "Sarcoscypha coccinea", "hr": "Crveni peharček", "shape": "funnel", "nc": false}, {"id": "b-scutellinia-scutellata", "lat": "Scutellinia scutellata", "hr": "Štitasta tanjurica", "shape": "coral", "nc": false}, {"id": "b-paxina-acetabulum", "lat": "Paxina acetabulum", "hr": "Proljetna rebrašica", "shape": "coral", "nc": false}, {"id": "b-peziza-vesiculosa", "lat": "Peziza vesiculosa", "hr": "Mjehurasta zdjeličarka", "shape": "funnel", "nc": false}, {"id": "b-peziza-badia", "lat": "Peziza badia", "hr": "Kestenjasta zdjeličarka", "shape": "funnel", "nc": false}, {"id": "b-discina-perlata", "lat": "Discina perlata", "hr": "Discina perlata", "shape": "coral", "nc": false}, {"id": "b-disciotis-venosa", "lat": "Disciotis venosa", "hr": "Žilasta tavica", "shape": "coral", "nc": false}, {"id": "b-sclerotinia-tuberosa", "lat": "Sclerotinia tuberosa", "hr": "Šumaričin pehar", "shape": "coral", "nc": false}, {"id": "r-agaricus-fuscofibrillosus", "lat": "Agaricus fuscofibrillosus", "hr": "Tamnovlaknata pečurka", "shape": "convex", "nc": false}, {"id": "r-aleuria-boudieri", "lat": "Aleuria boudieri", "hr": "Jarkocrvena tanjurica", "shape": "funnel", "nc": false}, {"id": "r-aleuria-flavorubens", "lat": "Aleuria flavorubens", "hr": "Žutocrvena tanjurica", "shape": "funnel", "nc": false}, {"id": "r-aleurocystidiellum-subcruentatum", "lat": "Aleurocystidiellum subcruentatum", "hr": "Crnogorična zdjelokoška", "shape": "coral", "nc": false}, {"id": "r-amanita-friabilis", "lat": "Amanita friabilis", "hr": "Johina preslica", "shape": "convex", "nc": false}, {"id": "r-amanita-lepiotoides", "lat": "Amanita lepiotoides", "hr": "Suncobranasta pupavka", "shape": "convex", "nc": false}, {"id": "r-amanita-pachyvolvata", "lat": "Amanita pachyvolvata", "hr": "Debeloovojna preslica", "shape": "convex", "nc": false}, {"id": "r-amanita-solitaria", "lat": "Amanita solitaria", "hr": "Ježasta muhara", "shape": "convex", "nc": false}, {"id": "r-amanita-vittadinii", "lat": "Amanita vittadinii", "hr": "Kuštrava muhara", "shape": "convex", "nc": false}, {"id": "r-amylocorticium-subincarnatum", "lat": "Amylocorticium subincarnatum", "hr": "Crvenkasta jodokorka", "shape": "coral", "nc": false}, {"id": "r-amylocystis-lapponicus", "lat": "Amylocystis lapponicus", "hr": "Prašumska sočnoporka", "shape": "coral", "nc": false}, {"id": "r-anomoporia-bombycina", "lat": "Anomoporia bombycina", "hr": "Svilenkasta koroporka", "shape": "coral", "nc": false}, {"id": "r-antrodia-crassa", "lat": "Antrodia crassa", "hr": "Debela drvoporka", "shape": "coral", "nc": false}, {"id": "r-antrodia-variiformis", "lat": "Antrodia variiformis", "hr": "Promjenjiva drvoporka", "shape": "coral", "nc": false}, {"id": "r-antrodiella-citrinella", "lat": "Antrodiella citrinella", "hr": "Limunasta smolocijevka", "shape": "coral", "nc": false}, {"id": "r-antrodiella-fragrans", "lat": "Antrodiella fragrans", "hr": "Mirisna smolocijevka", "shape": "coral", "nc": false}, {"id": "r-arrhenia-spathulata", "lat": "Arrhenia spathulata", "hr": "Lopatasta jezičarka", "shape": "coral", "nc": false}, {"id": "r-ascobolus-behnitziensis", "lat": "Ascobolus behnitziensis", "hr": "Raspucanosporna balegarka", "shape": "coral", "nc": false}, {"id": "r-ascobolus-epimyces", "lat": "Ascobolus epimyces", "hr": "Maslinasta balegarka", "shape": "coral", "nc": false}, {"id": "r-ascotremella-faginea", "lat": "Ascotremella faginea", "hr": "Bukova lažidrhtalica", "shape": "coral", "nc": false}, {"id": "r-atheloderma-mirabile", "lat": "Atheloderma mirabile", "hr": "Čudesna nitokoška", "shape": "coral", "nc": false}, {"id": "r-auriporia-aurulenta", "lat": "Auriporia aurulenta", "hr": "Narančasta zlatoporka", "shape": "coral", "nc": false}, {"id": "r-baeospora-myriadophylla", "lat": "Baeospora myriadophylla", "hr": "Zimska sitnosporka", "shape": "coral", "nc": false}, {"id": "r-bisporella-calycellinoides", "lat": "Bisporella calycellinoides", "hr": "Ružina glatkočaška", "shape": "coral", "nc": false}, {"id": "r-bolbitius-lacteus", "lat": "Bolbitius lacteus", "hr": "Bijeli smetlištar", "shape": "conical", "nc": false}, {"id": "r-boletus-adonis", "lat": "Boletus adonis", "hr": "Creski vrganj", "shape": "convex", "nc": false}, {"id": "r-boletus-depilatus", "lat": "Boletus depilatus", "hr": "Glatki vrganj", "shape": "convex", "nc": false}, {"id": "r-boletus-torosus", "lat": "Boletus torosus", "hr": "Teški vrganj", "shape": "convex", "nc": false}, {"id": "r-callistosporium-donadinii", "lat": "Callistosporium donadinii", "hr": "Brončana zelenčica", "shape": "coral", "nc": false}, {"id": "r-callistosporium-elaeodes", "lat": "Callistosporium elaeodes", "hr": "Bjelogorična zelenčica", "shape": "coral", "nc": false}, {"id": "r-callistosporium-luteoolivaceum", "lat": "Callistosporium luteoolivaceum", "hr": "Gustolisna zelenčica", "shape": "coral", "nc": false}, {"id": "r-calocybe-carnea", "lat": "Calocybe carnea", "hr": "Ružičasta ljepoglavka", "shape": "convex", "nc": false}, {"id": "r-calocybe-constricta", "lat": "Calocybe constricta", "hr": "Prstenasta ljepoglavka", "shape": "convex", "nc": false}, {"id": "r-calocybe-ionides", "lat": "Calocybe ionides", "hr": "Ljubičasta ljepoglavka", "shape": "convex", "nc": false}, {"id": "r-calocybe-obscurissima", "lat": "Calocybe obscurissima", "hr": "Tamna ljepoglavka", "shape": "convex", "nc": false}, {"id": "r-camarophyllopsis-foetens", "lat": "Camarophyllopsis foetens", "hr": "Mirisna prljavica", "shape": "coral", "nc": false}, {"id": "r-camarophyllopsis-micacea", "lat": "Camarophyllopsis micacea", "hr": "Žutosmeđa prljavica", "shape": "coral", "nc": false}, {"id": "r-camarophyllopsis-phaeophylla", "lat": "Camarophyllopsis phaeophylla", "hr": "Smeđelisna prljavica", "shape": "coral", "nc": false}, {"id": "r-campanella-caesia", "lat": "Campanella caesia", "hr": "Plavkasta zvončica", "shape": "coral", "nc": false}, {"id": "r-ceriporiopsis-jelicii", "lat": "Ceriporiopsis jelicii", "hr": "Plitvička krastoporka", "shape": "coral", "nc": false}, {"id": "r-ceriporiopsis-myceliosa", "lat": "Ceriporiopsis myceliosa", "hr": "Korjenasta krastoporka", "shape": "coral", "nc": false}, {"id": "r-ceriporiopsis-subrufa", "lat": "Ceriporiopsis subrufa", "hr": "Crveneća krastoporka", "shape": "coral", "nc": false}, {"id": "r-chalciporus-rubinus", "lat": "Chalciporus rubinus", "hr": "Rubinski ljepljivac", "shape": "convex", "nc": false}, {"id": "r-cheilymenia-magnifica", "lat": "Cheilymenia magnifica", "hr": "Čaškasta trepavčica", "shape": "coral", "nc": false}, {"id": "r-ciboria-aestivalis", "lat": "Ciboria aestivalis", "hr": "Ljetna peharica", "shape": "coral", "nc": false}, {"id": "r-clavulicium-macounii", "lat": "Clavulicium macounii", "hr": "Smeđasta voskokorka", "shape": "coral", "nc": false}, {"id": "r-clitocybe-collina", "lat": "Clitocybe collina", "hr": "Stepska uleknjača", "shape": "funnel", "nc": false}, {"id": "r-clitocybula-lacerata", "lat": "Clitocybula lacerata", "hr": "Raspucana vlaknatica", "shape": "coral", "nc": false}, {"id": "r-contumyces-rosellus", "lat": "Contumyces rosellus", "hr": "Ružičasta mahovinka", "shape": "coral", "nc": false}, {"id": "r-coprinus-strossmayeri", "lat": "Coprinus strossmayeri", "hr": "Slavonska gnojištarka", "shape": "coral", "nc": false}, {"id": "r-cordyceps-bifusispora", "lat": "Cordyceps bifusispora", "hr": "Narančastožuta batinica", "shape": "coral", "nc": false}, {"id": "r-cordyceps-intermedia", "lat": "Cordyceps intermedia", "hr": "Tamnoglava batinica", "shape": "coral", "nc": false}, {"id": "r-cordyceps-longisegmentis", "lat": "Cordyceps longisegmentis", "hr": "Zrnatonoga batinica", "shape": "coral", "nc": false}, {"id": "r-cordyceps-riverae", "lat": "Cordyceps riverae", "hr": "Špiljska batinica", "shape": "coral", "nc": false}, {"id": "r-cordyceps-sphecocephala", "lat": "Cordyceps sphecocephala", "hr": "Vitka batinica", "shape": "coral", "nc": false}, {"id": "r-cortinarius-bulliardii", "lat": "Cortinarius bulliardii", "hr": "Plamenonoga koprenka", "shape": "convex", "nc": false}, {"id": "r-cortinarius-ionochlorus", "lat": "Cortinarius ionochlorus", "hr": "Ljubičasto-zelena koprenka", "shape": "convex", "nc": false}, {"id": "r-cotylidia-pannosa", "lat": "Cotylidia pannosa", "hr": "Valovita zvrčica", "shape": "coral", "nc": false}, {"id": "r-crinipellis-subtomentosa", "lat": "Crinipellis subtomentosa", "hr": "Primorska dlakavica", "shape": "coral", "nc": false}, {"id": "r-cudonia-circinans", "lat": "Cudonia circinans", "hr": "Okruglasta kožoglavka", "shape": "coral", "nc": false}, {"id": "r-cystoderma-superbum", "lat": "Cystoderma superbum", "hr": "Grimizna hrapavica", "shape": "small", "nc": false}, {"id": "r-dacryobolus-karstenii", "lat": "Dacryobolus karstenii", "hr": "Suha zubokorka", "shape": "coral", "nc": false}, {"id": "r-dentipellis-fragilis", "lat": "Dentipellis fragilis", "hr": "Krhka iglokoška", "shape": "coral", "nc": false}, {"id": "r-dermoloma-cuneifolium", "lat": "Dermoloma cuneifolium", "hr": "Velika besjajnica", "shape": "coral", "nc": false}, {"id": "r-dermoloma-josserandii", "lat": "Dermoloma josserandii", "hr": "Sitnosporna besjajnica", "shape": "coral", "nc": false}, {"id": "r-dermoloma-pseudocuneifolium", "lat": "Dermoloma pseudocuneifolium", "hr": "Mala besjajnica", "shape": "coral", "nc": false}, {"id": "r-desmazierella-acicola", "lat": "Desmazierella acicola", "hr": "Borova bodljočaška", "shape": "coral", "nc": false}, {"id": "r-discina-fastigiata", "lat": "Discina fastigiata", "hr": "Grebenasta pločarica", "shape": "coral", "nc": false}, {"id": "r-discina-leucoxantha", "lat": "Discina leucoxantha", "hr": "Blijedožuta pločarica", "shape": "coral", "nc": false}, {"id": "r-discina-montana", "lat": "Discina montana", "hr": "Planinska pločarica", "shape": "coral", "nc": false}, {"id": "r-discina-parma", "lat": "Discina parma", "hr": "Štitasta pločarica", "shape": "coral", "nc": false}, {"id": "r-disciseda-bovista", "lat": "Disciseda bovista", "hr": "Velika okretnica", "shape": "coral", "nc": false}, {"id": "r-disciseda-candida", "lat": "Disciseda candida", "hr": "Mala okretnica", "shape": "coral", "nc": false}, {"id": "r-elaphomyces-anthracinus", "lat": "Elaphomyces anthracinus", "hr": "Crna jelen-gljiva", "shape": "coral", "nc": false}, {"id": "r-endoptychum-agaricoides", "lat": "Endoptychum agaricoides", "hr": "Stepska stožerka", "shape": "coral", "nc": false}, {"id": "r-entoloma-bloxamii", "lat": "Entoloma bloxamii", "hr": "Ljubičastoplava rudoliska", "shape": "convex", "nc": false}, {"id": "r-entoloma-caesiocinctum", "lat": "Entoloma caesiocinctum", "hr": "Plavorubna rudoliska", "shape": "convex", "nc": false}, {"id": "r-entoloma-catalaunicum", "lat": "Entoloma catalaunicum", "hr": "Planinska rudoliska", "shape": "convex", "nc": false}, {"id": "r-entoloma-corvinum", "lat": "Entoloma corvinum", "hr": "Vranska rudoliska", "shape": "convex", "nc": false}, {"id": "r-entoloma-cyanulum", "lat": "Entoloma cyanulum", "hr": "Patuljasta rudoliska", "shape": "convex", "nc": false}, {"id": "r-entoloma-kervernii", "lat": "Entoloma kervernii", "hr": "Smećkastočehava rudoliska", "shape": "convex", "nc": false}, {"id": "r-entoloma-opacum", "lat": "Entoloma opacum", "hr": "Mračna rudoliska", "shape": "convex", "nc": false}, {"id": "r-entoloma-plebeioides", "lat": "Entoloma plebeioides", "hr": "Slavonska rudoliska", "shape": "convex", "nc": false}, {"id": "r-entoloma-plebejum", "lat": "Entoloma plebejum", "hr": "Rana rudoliska", "shape": "convex", "nc": false}, {"id": "r-entoloma-porphyrophaeum", "lat": "Entoloma porphyrophaeum", "hr": "Grimiznosmeđa rudoliska", "shape": "convex", "nc": false}, {"id": "r-entoloma-pseudoturci", "lat": "Entoloma pseudoturci", "hr": "Sitnočehava rudoliska", "shape": "convex", "nc": false}, {"id": "r-entoloma-reinwaldii", "lat": "Entoloma reinwaldii", "hr": "Ljupka rudoliska", "shape": "convex", "nc": false}, {"id": "r-entoloma-rhombisporum", "lat": "Entoloma rhombisporum", "hr": "Rombosporna rudoliska", "shape": "convex", "nc": false}, {"id": "r-entoloma-saundersii", "lat": "Entoloma saundersii", "hr": "Ožujska rudoliska", "shape": "convex", "nc": false}, {"id": "r-faerberia-carbonaria", "lat": "Faerberia carbonaria", "hr": "Žilasti zgarištar", "shape": "coral", "nc": false}, {"id": "r-flammulina-ononidis", "lat": "Flammulina ononidis", "hr": "Travnjačka panjevka", "shape": "convex", "nc": false}, {"id": "r-flavophlebia-sulphureoisabellina", "lat": "Flavophlebia sulphureoisabellina", "hr": "Sumporasta žutožilnica", "shape": "coral", "nc": false}, {"id": "r-fomitopsis-rosea", "lat": "Fomitopsis rosea", "hr": "Ružičasta gubovka", "shape": "shelf", "nc": false}, {"id": "r-fomitopsis-spraguei", "lat": "Fomitopsis spraguei", "hr": "Kestenova gubovka", "shape": "shelf", "nc": false}, {"id": "r-galerina-jaapii", "lat": "Galerina jaapii", "hr": "Ovjenčana patuljica", "shape": "small", "nc": false}, {"id": "r-galerina-paludosa", "lat": "Galerina paludosa", "hr": "Močvarna patuljica", "shape": "small", "nc": false}, {"id": "r-galerina-tibiicystis", "lat": "Galerina tibiicystis", "hr": "Cretna patuljica", "shape": "small", "nc": false}, {"id": "r-gastrosporium-simplex", "lat": "Gastrosporium simplex", "hr": "Podzemna travarica", "shape": "coral", "nc": false}, {"id": "r-geastrum-minimum", "lat": "Geastrum minimum", "hr": "Sitna zvjezdača", "shape": "ball", "nc": false}, {"id": "r-geoglossum-cookeanum", "lat": "Geoglossum cookeanum", "hr": "Travnjački jezičnjak", "shape": "coral", "nc": false}, {"id": "r-geoglossum-glutinosum", "lat": "Geoglossum glutinosum", "hr": "Ljepljivi jezičnjak", "shape": "coral", "nc": false}, {"id": "r-geoglossum-umbratile", "lat": "Geoglossum umbratile", "hr": "Tamni jezičnjak", "shape": "coral", "nc": false}, {"id": "r-geopora-nicaeensis", "lat": "Geopora nicaeensis", "hr": "Primorska zemljoporka", "shape": "coral", "nc": false}, {"id": "r-geopyxis-majalis", "lat": "Geopyxis majalis", "hr": "Svibanjska žutočaška", "shape": "coral", "nc": false}, {"id": "r-gerhardtia-piperata", "lat": "Gerhardtia piperata", "hr": "Paprena kostanjevčica", "shape": "coral", "nc": false}, {"id": "r-gloeocystidiellum-subasperisporum", "lat": "Gloeocystidiellum subasperisporum", "hr": "Trnosporna voskokoška", "shape": "coral", "nc": false}, {"id": "r-gomphidius-roseus", "lat": "Gomphidius roseus", "hr": "Ružičasti slinar", "shape": "convex", "nc": false}, {"id": "r-gymnopilus-bellulus", "lat": "Gymnopilus bellulus", "hr": "Lijepa plamenka", "shape": "convex", "nc": false}, {"id": "r-gymnopilus-picreus", "lat": "Gymnopilus picreus", "hr": "Gorka plamenka", "shape": "convex", "nc": false}, {"id": "r-haasiella-splendidissima", "lat": "Haasiella splendidissima", "hr": "Sjajna prekrasnica", "shape": "coral", "nc": false}, {"id": "r-haasiella-venustissima", "lat": "Haasiella venustissima", "hr": "Ljupka prekrasnica", "shape": "coral", "nc": false}, {"id": "r-hapalopilus-croceus", "lat": "Hapalopilus croceus", "hr": "Šafranasta mekoporka", "shape": "shelf", "nc": false}, {"id": "r-hapalopilus-salmonicolor", "lat": "Hapalopilus salmonicolor", "hr": "Crnogorična mekoporka", "shape": "shelf", "nc": false}, {"id": "r-hebeloma-gigaspermum", "lat": "Hebeloma gigaspermum", "hr": "Velikosporna bljedunjavka", "shape": "convex", "nc": false}, {"id": "r-helvella-albella", "lat": "Helvella albella", "hr": "Dvobojni hrčak", "shape": "tall", "nc": false}, {"id": "r-helvella-branzeziana", "lat": "Helvella branzeziana", "hr": "Kapičasti hrčak", "shape": "tall", "nc": false}, {"id": "r-helvella-cupuliformis", "lat": "Helvella cupuliformis", "hr": "Peharasti hrčak", "shape": "tall", "nc": false}, {"id": "r-helvella-helvellula", "lat": "Helvella helvellula", "hr": "Sitni hrčak", "shape": "tall", "nc": false}, {"id": "r-helvella-lactea", "lat": "Helvella lactea", "hr": "Mliječni hrčak", "shape": "tall", "nc": false}, {"id": "r-helvella-phlebophora", "lat": "Helvella phlebophora", "hr": "Listićavi hrčak", "shape": "tall", "nc": false}, {"id": "r-helvella-queletiana", "lat": "Helvella queletiana", "hr": "Žilasti hrčak", "shape": "tall", "nc": false}, {"id": "r-helvella-spadicea", "lat": "Helvella spadicea", "hr": "Crno-bijeli hrčak", "shape": "tall", "nc": false}, {"id": "r-hericium-flagellum", "lat": "Hericium flagellum", "hr": "Jelov igličar", "shape": "coral", "nc": false}, {"id": "r-hexagonia-nitida", "lat": "Hexagonia nitida", "hr": "Šesterokutna velikoporka", "shape": "coral", "nc": false}, {"id": "r-heyderia-abietis", "lat": "Heyderia abietis", "hr": "Jelova kapuljica", "shape": "coral", "nc": false}, {"id": "r-holwaya-mucida", "lat": "Holwaya mucida", "hr": "Sluzava dvoličnica", "shape": "coral", "nc": false}, {"id": "r-humaria-aurantia", "lat": "Humaria aurantia", "hr": "Narančasta čekinjarica", "shape": "coral", "nc": false}, {"id": "r-hydropus-atramentosus", "lat": "Hydropus atramentosus", "hr": "Crneća vodonoška", "shape": "coral", "nc": false}, {"id": "r-hygrocybe-calciphila", "lat": "Hygrocybe calciphila", "hr": "Vapnenačka vlažnica", "shape": "conical", "nc": false}, {"id": "r-hygrocybe-calyptriformis", "lat": "Hygrocybe calyptriformis", "hr": "Ružičasta vlažnica", "shape": "conical", "nc": false}, {"id": "r-hygrocybe-citrinovirens", "lat": "Hygrocybe citrinovirens", "hr": "Zelenkastožuta vlažnica", "shape": "conical", "nc": false}, {"id": "r-hygrocybe-coccineocrenata", "lat": "Hygrocybe coccineocrenata", "hr": "Cretna vlažnica", "shape": "conical", "nc": false}, {"id": "r-hygrocybe-colemanniana", "lat": "Hygrocybe colemanniana", "hr": "Smeđa vlažnica", "shape": "conical", "nc": false}, {"id": "r-hygrocybe-fornicata", "lat": "Hygrocybe fornicata", "hr": "Svijetlosmeđa vlažnica", "shape": "conical", "nc": false}, {"id": "r-hygrocybe-ingrata", "lat": "Hygrocybe ingrata", "hr": "Crveneća vlažnica", "shape": "conical", "nc": false}, {"id": "r-hygrocybe-intermedia", "lat": "Hygrocybe intermedia", "hr": "Suha vlažnica", "shape": "conical", "nc": false}, {"id": "r-hygrocybe-irrigata", "lat": "Hygrocybe irrigata", "hr": "Sluzava vlažnica", "shape": "conical", "nc": false}, {"id": "r-hygrocybe-lacmus", "lat": "Hygrocybe lacmus", "hr": "Ljubičastosiva vlažnica", "shape": "conical", "nc": false}, {"id": "r-hygrocybe-perplexa", "lat": "Hygrocybe perplexa", "hr": "Ciglasta vlažnica", "shape": "conical", "nc": false}, {"id": "r-hygrocybe-russocoriacea", "lat": "Hygrocybe russocoriacea", "hr": "Mirisna vlažnica", "shape": "conical", "nc": false}, {"id": "r-hygrocybe-sciophanoides", "lat": "Hygrocybe sciophanoides", "hr": "Lilasta vlažnica", "shape": "conical", "nc": false}, {"id": "r-hygrocybe-spadicea", "lat": "Hygrocybe spadicea", "hr": "Žuto-smeđa vlažnica", "shape": "conical", "nc": false}, {"id": "r-hygrocybe-splendidissima", "lat": "Hygrocybe splendidissima", "hr": "Sjajna vlažnica", "shape": "conical", "nc": false}, {"id": "r-hygrocybe-subpapillata", "lat": "Hygrocybe subpapillata", "hr": "Grbičasta vlažnica", "shape": "conical", "nc": false}, {"id": "r-hygrophorus-calophyllus", "lat": "Hygrophorus calophyllus", "hr": "Ljepolisna puževica", "shape": "convex", "nc": false}, {"id": "r-hygrophorus-camarophyllus", "lat": "Hygrophorus camarophyllus", "hr": "Sivosmeđa puževica", "shape": "convex", "nc": false}, {"id": "r-hygrophorus-capreolarius", "lat": "Hygrophorus capreolarius", "hr": "Vinskicrvena puževica", "shape": "convex", "nc": false}, {"id": "r-hygrophorus-hyacinthinus", "lat": "Hygrophorus hyacinthinus", "hr": "Zumbulova puževica", "shape": "convex", "nc": false}, {"id": "r-hygrophorus-hypothejus", "lat": "Hygrophorus hypothejus", "hr": "Kasna puževica", "shape": "convex", "nc": false}, {"id": "r-hygrophorus-mesotephrus", "lat": "Hygrophorus mesotephrus", "hr": "Smeđe-bijela puževica", "shape": "convex", "nc": false}, {"id": "r-hyphoderma-cremeoalbum", "lat": "Hyphoderma cremeoalbum", "hr": "Žućkastobijela nitokoška", "shape": "coral", "nc": false}, {"id": "r-hyphoderma-guttuliferum", "lat": "Hyphoderma guttuliferum", "hr": "Kapljičasta nitokoška", "shape": "coral", "nc": false}, {"id": "r-hyphoderma-litschaueri", "lat": "Hyphoderma litschaueri", "hr": "Američka nitokoška", "shape": "coral", "nc": false}, {"id": "r-hyphoderma-macedonicum", "lat": "Hyphoderma macedonicum", "hr": "Makedonska nitokoška", "shape": "coral", "nc": false}, {"id": "r-hypholoma-elongatum", "lat": "Hypholoma elongatum", "hr": "Cretna sumporača", "shape": "convex", "nc": false}, {"id": "r-hypholoma-myosotis", "lat": "Hypholoma myosotis", "hr": "Ljepljiva sumporača", "shape": "convex", "nc": false}, {"id": "r-inonotus-dryophilus", "lat": "Inonotus dryophilus", "hr": "Hrastov kopitar", "shape": "shelf", "nc": false}, {"id": "r-inonotus-nidus-pici", "lat": "Inonotus nidus-pici", "hr": "Dupljarski kopitar", "shape": "shelf", "nc": false}, {"id": "r-jaapia-ochroleuca", "lat": "Jaapia ochroleuca", "hr": "Blijedosmeđa krupičavka", "shape": "coral", "nc": false}, {"id": "r-junghuhnia-collabens", "lat": "Junghuhnia collabens", "hr": "Cimetasta porokoška", "shape": "coral", "nc": false}, {"id": "r-junghuhnia-fimbriatella", "lat": "Junghuhnia fimbriatella", "hr": "Resasta porokoška", "shape": "coral", "nc": false}, {"id": "r-kuehneromyces-lignicola", "lat": "Kuehneromyces lignicola", "hr": "Proljetna panjevčica", "shape": "convex", "nc": false}, {"id": "r-laccaria-pumila", "lat": "Laccaria pumila", "hr": "Močvarna lakovka", "shape": "convex", "nc": false}, {"id": "r-lactarius-acris", "lat": "Lactarius acris", "hr": "Oštra mliječnica", "shape": "funnel", "nc": false}, {"id": "r-lactarius-cistophilus", "lat": "Lactarius cistophilus", "hr": "Bušinova mliječnica", "shape": "funnel", "nc": false}, {"id": "r-lactarius-lacunarum", "lat": "Lactarius lacunarum", "hr": "Močvarna mliječnica", "shape": "funnel", "nc": false}, {"id": "r-lactarius-lilacinus", "lat": "Lactarius lilacinus", "hr": "Lilasta mliječnica", "shape": "funnel", "nc": false}, {"id": "r-lactarius-omphaliformis", "lat": "Lactarius omphaliformis", "hr": "Ljevkasta mliječnica", "shape": "funnel", "nc": false}, {"id": "r-lactarius-tesquorum", "lat": "Lactarius tesquorum", "hr": "Ružičasta mliječnica", "shape": "funnel", "nc": false}, {"id": "r-lambertella-corni-maris", "lat": "Lambertella corni-maris", "hr": "Drijenova smeđetrusnica", "shape": "coral", "nc": false}, {"id": "r-lentinus-degener", "lat": "Lentinus degener", "hr": "Uskolisna pilašica", "shape": "funnel", "nc": false}, {"id": "r-leptosporomyces-roseus", "lat": "Leptosporomyces roseus", "hr": "Ružičasta kožičarka", "shape": "coral", "nc": false}, {"id": "r-leucoagaricus-pilatianus", "lat": "Leucoagaricus pilatianus", "hr": "Crvenosmeđa pečurkovica", "shape": "umbrella-s", "nc": false}, {"id": "r-leucogaster-nudus", "lat": "Leucogaster nudus", "hr": "Glatka bjelotrbuška", "shape": "coral", "nc": false}, {"id": "r-leucopaxillus-compactus", "lat": "Leucopaxillus compactus", "hr": "Trobojna podvijenka", "shape": "funnel", "nc": false}, {"id": "r-leucopaxillus-macrocephalus", "lat": "Leucopaxillus macrocephalus", "hr": "Glavata podvijenka", "shape": "funnel", "nc": false}, {"id": "r-lyophyllum-rhopalopodium", "lat": "Lyophyllum rhopalopodium", "hr": "Debelonoga kostanjevčica", "shape": "convex", "nc": false}, {"id": "r-lyophyllum-transforme", "lat": "Lyophyllum transforme", "hr": "Trokutosporna kostanjevčica", "shape": "convex", "nc": false}, {"id": "r-marasmiellus-humillimus", "lat": "Marasmiellus humillimus", "hr": "Patuljasta posušnica", "shape": "coral", "nc": false}, {"id": "r-marasmius-anomalus", "lat": "Marasmius anomalus", "hr": "Travnjačka pritajnica", "shape": "small", "nc": false}, {"id": "r-marasmius-buxi", "lat": "Marasmius buxi", "hr": "Šimširova pritajnica", "shape": "small", "nc": false}, {"id": "r-marasmius-caricis", "lat": "Marasmius caricis", "hr": "Šaševa pritajnica", "shape": "small", "nc": false}, {"id": "r-marasmius-collinus", "lat": "Marasmius collinus", "hr": "Krhkonoga pritajnica", "shape": "small", "nc": false}, {"id": "r-marasmius-cornelii", "lat": "Marasmius cornelii", "hr": "Ljutkova pritajnica", "shape": "small", "nc": false}, {"id": "r-marasmius-hudsonii", "lat": "Marasmius hudsonii", "hr": "Igličava pritajnica", "shape": "small", "nc": false}, {"id": "r-marasmius-menieri", "lat": "Marasmius menieri", "hr": "Rogozova pritajnica", "shape": "small", "nc": false}, {"id": "r-marasmius-undatus", "lat": "Marasmius undatus", "hr": "Bujadna pritajnica", "shape": "small", "nc": false}, {"id": "r-melanogaster-vittadinii", "lat": "Melanogaster vittadinii", "hr": "Jajosporna crnotrbuška", "shape": "coral", "nc": false}, {"id": "r-metulodontia-nivea", "lat": "Metulodontia nivea", "hr": "Bijela grbavičarka", "shape": "coral", "nc": false}, {"id": "r-microglossum-olivaceum", "lat": "Microglossum olivaceum", "hr": "Maslinasti jezičak", "shape": "coral", "nc": false}, {"id": "r-microglossum-viride", "lat": "Microglossum viride", "hr": "Zeleni jezičak", "shape": "coral", "nc": false}, {"id": "r-mitrula-paludosa", "lat": "Mitrula paludosa", "hr": "Močvarna kapuljača", "shape": "coral", "nc": false}, {"id": "r-moellerodiscus-iodotingens", "lat": "Moellerodiscus iodotingens", "hr": "Primorska peharica", "shape": "coral", "nc": false}, {"id": "r-moellerodiscus-tenuistipes", "lat": "Moellerodiscus tenuistipes", "hr": "Tankonoga peharica", "shape": "coral", "nc": false}, {"id": "r-mollisia-olivascens", "lat": "Mollisia olivascens", "hr": "Maslinasta pločica", "shape": "coral", "nc": false}, {"id": "r-mucronella-bresadolae", "lat": "Mucronella bresadolae", "hr": "Duga igličarka", "shape": "coral", "nc": false}, {"id": "r-mucronella-calva", "lat": "Mucronella calva", "hr": "Kratka igličarka", "shape": "coral", "nc": false}, {"id": "r-mycena-latifolia", "lat": "Mycena latifolia", "hr": "Širokolisna šljemovka", "shape": "small", "nc": false}, {"id": "r-mycenastrum-corium", "lat": "Mycenastrum corium", "hr": "Kožasta oklopnica", "shape": "coral", "nc": false}, {"id": "r-myriosclerotinia-dennisii", "lat": "Myriosclerotinia dennisii", "hr": "Suhoperkina močvarnica", "shape": "coral", "nc": false}, {"id": "r-myriostoma-coliforme", "lat": "Myriostoma coliforme", "hr": "Zvjezdasta sitarica", "shape": "coral", "nc": false}, {"id": "r-oligoporus-cerifluus", "lat": "Oligoporus cerifluus", "hr": "Sitna maloporka", "shape": "coral", "nc": false}, {"id": "r-oligoporus-lowei", "lat": "Oligoporus lowei", "hr": "Pojasasta maloporka", "shape": "coral", "nc": false}, {"id": "r-oligoporus-placenta", "lat": "Oligoporus placenta", "hr": "Ružičasta maloporka", "shape": "coral", "nc": false}, {"id": "r-ombrophila-rivulorum", "lat": "Ombrophila rivulorum", "hr": "Potočna vlagoljupka", "shape": "coral", "nc": false}, {"id": "r-omphalina-baeospora", "lat": "Omphalina baeospora", "hr": "Sitnosporna ljevčica", "shape": "funnel", "nc": false}, {"id": "r-omphalina-chrysophylla", "lat": "Omphalina chrysophylla", "hr": "Zlatolisna ljevčica", "shape": "funnel", "nc": false}, {"id": "r-omphalina-obatra", "lat": "Omphalina obatra", "hr": "Mrka ljevčica", "shape": "funnel", "nc": false}, {"id": "r-onygena-equina", "lat": "Onygena equina", "hr": "Rožna stapkoglavka", "shape": "coral", "nc": false}, {"id": "r-orbilia-polyspora", "lat": "Orbilia polyspora", "hr": "Mnogosporna voštanica", "shape": "coral", "nc": false}, {"id": "r-ossicaulis-lignatilis", "lat": "Ossicaulis lignatilis", "hr": "Drvna uleknica", "shape": "coral", "nc": false}, {"id": "r-otidea-propinquata", "lat": "Otidea propinquata", "hr": "Jelova uška", "shape": "ear", "nc": false}, {"id": "r-pachyella-peltata", "lat": "Pachyella peltata", "hr": "Štitasta vodoljupka", "shape": "coral", "nc": false}, {"id": "r-pachyella-pseudosuccosa", "lat": "Pachyella pseudosuccosa", "hr": "Sočna vodoljupka", "shape": "coral", "nc": false}, {"id": "r-pachyella-punctispora", "lat": "Pachyella punctispora", "hr": "Tamna vodoljupka", "shape": "coral", "nc": false}, {"id": "r-pachyella-violaceonigra", "lat": "Pachyella violaceonigra", "hr": "Ljubičastocrna vodoljupka", "shape": "coral", "nc": false}, {"id": "r-parmastomyces-mollissimus", "lat": "Parmastomyces mollissimus", "hr": "Promjenjiva priljepnica", "shape": "coral", "nc": false}, {"id": "r-peziza-alborosea", "lat": "Peziza alborosea", "hr": "Blijedoružičasta zdjeličarka", "shape": "funnel", "nc": false}, {"id": "r-peziza-ammophila", "lat": "Peziza ammophila", "hr": "Pješčarska zdjeličarka", "shape": "funnel", "nc": false}, {"id": "r-peziza-apiculata", "lat": "Peziza apiculata", "hr": "Šiljatosporna zdjeličarka", "shape": "funnel", "nc": false}, {"id": "r-peziza-boltonii", "lat": "Peziza boltonii", "hr": "Tamnozrnata zdjeličarka", "shape": "funnel", "nc": false}, {"id": "r-peziza-buxea", "lat": "Peziza buxea", "hr": "Slamnata zdjeličarka", "shape": "funnel", "nc": false}, {"id": "r-peziza-flavida", "lat": "Peziza flavida", "hr": "Žuta zdjeličarka", "shape": "funnel", "nc": false}, {"id": "r-peziza-merdae", "lat": "Peziza merdae", "hr": "Izmetna zdjeličarka", "shape": "funnel", "nc": false}, {"id": "r-peziza-muscicola", "lat": "Peziza muscicola", "hr": "Mahovinska zdjeličarka", "shape": "funnel", "nc": false}, {"id": "r-peziza-nivalis", "lat": "Peziza nivalis", "hr": "Snježna zdjeličarka", "shape": "funnel", "nc": false}, {"id": "r-peziza-obtusapiculata", "lat": "Peziza obtusapiculata", "hr": "Panjeva zdjeličarka", "shape": "funnel", "nc": false}, {"id": "r-peziza-saccardiana", "lat": "Peziza saccardiana", "hr": "Riđezrnata zdjeličarka", "shape": "funnel", "nc": false}, {"id": "r-peziza-subuliginosa", "lat": "Peziza subuliginosa", "hr": "Cretna zdjeličarka", "shape": "funnel", "nc": false}, {"id": "r-peziza-subumbrina", "lat": "Peziza subumbrina", "hr": "Tamnosmeđa zdjeličarka", "shape": "funnel", "nc": false}, {"id": "r-phaeocollybia-jennyi", "lat": "Phaeocollybia jennyi", "hr": "Narančastosmeđa šiljatonoška", "shape": "coral", "nc": false}, {"id": "r-phaeomarasmius-limulatus", "lat": "Phaeomarasmius limulatus", "hr": "Smeđežuta zrnoglavka", "shape": "coral", "nc": false}, {"id": "r-phaeomarasmius-muricatus", "lat": "Phaeomarasmius muricatus", "hr": "Bodljikava zrnoglavka", "shape": "coral", "nc": false}, {"id": "r-phallogaster-saccatus", "lat": "Phallogaster saccatus", "hr": "Vrećasta strškovica", "shape": "coral", "nc": false}, {"id": "r-phellinus-nigrolimitatus", "lat": "Phellinus nigrolimitatus", "hr": "Crnoobrubljeni plutnjak", "shape": "shelf", "nc": false}, {"id": "r-phellinus-pouzarii", "lat": "Phellinus pouzarii", "hr": "Pouzarov plutnjak", "shape": "shelf", "nc": false}, {"id": "r-phlebia-albida", "lat": "Phlebia albida", "hr": "Bjelkasta žilnica", "shape": "coral", "nc": false}, {"id": "r-phlebia-centrifuga", "lat": "Phlebia centrifuga", "hr": "Ljubičastosiva žilnica", "shape": "coral", "nc": false}, {"id": "r-phlebia-cornea", "lat": "Phlebia cornea", "hr": "Rožnata žilnica", "shape": "coral", "nc": false}, {"id": "r-phlebia-griseoflavescens", "lat": "Phlebia griseoflavescens", "hr": "Sivožućkasta žilnica", "shape": "coral", "nc": false}, {"id": "r-phlebia-subulata", "lat": "Phlebia subulata", "hr": "Prašumska žilnica", "shape": "coral", "nc": false}, {"id": "r-phylloporus-pelletieri", "lat": "Phylloporus pelletieri", "hr": "Crvenkasti listićavac", "shape": "coral", "nc": false}, {"id": "r-phyllotopsis-nidulans", "lat": "Phyllotopsis nidulans", "hr": "Gnjezdasti listar", "shape": "coral", "nc": false}, {"id": "r-piptoporus-quercinus", "lat": "Piptoporus quercinus", "hr": "Hrastova kopitarka", "shape": "shelf", "nc": false}, {"id": "r-piptoporus-soloniensis", "lat": "Piptoporus soloniensis", "hr": "Lepezasta kopitarka", "shape": "shelf", "nc": false}, {"id": "r-pisolithus-arhizus", "lat": "Pisolithus arhizus", "hr": "Velika graškovica", "shape": "coral", "nc": false}, {"id": "r-plectania-melastoma", "lat": "Plectania melastoma", "hr": "Crvenkasta crnočaška", "shape": "coral", "nc": false}, {"id": "r-plectania-platensis", "lat": "Plectania platensis", "hr": "Južnjačka crnočaška", "shape": "coral", "nc": false}, {"id": "r-pleurotus-calyptratus", "lat": "Pleurotus calyptratus", "hr": "Topolova krivonoška", "shape": "shelf", "nc": false}, {"id": "r-pleurotus-cornucopiae", "lat": "Pleurotus cornucopiae", "hr": "Brestovača", "shape": "shelf", "nc": false}, {"id": "r-poronia-punctata", "lat": "Poronia punctata", "hr": "Magareća točkarica", "shape": "coral", "nc": false}, {"id": "r-porphyrellus-porphyrosporus", "lat": "Porphyrellus porphyrosporus", "hr": "Čađavi tamnocijevac", "shape": "convex", "nc": false}, {"id": "r-psathyrella-ammophila", "lat": "Psathyrella ammophila", "hr": "Pješčarska slabunjavka", "shape": "conical", "nc": false}, {"id": "r-psathyrella-melanthina", "lat": "Psathyrella melanthina", "hr": "Siva slabunjavka", "shape": "conical", "nc": false}, {"id": "r-psathyrella-typhae", "lat": "Psathyrella typhae", "hr": "Rogozova slabunjavka", "shape": "conical", "nc": false}, {"id": "r-pseudoomphalina-kalchbrenneri", "lat": "Pseudoomphalina kalchbrenneri", "hr": "Stisnutonoga lažnoljevčica", "shape": "coral", "nc": false}, {"id": "r-pseudoplectania-melaena", "lat": "Pseudoplectania melaena", "hr": "Jelova crnočaška", "shape": "coral", "nc": false}, {"id": "r-pseudorhizina-sphaerospora", "lat": "Pseudorhizina sphaerospora", "hr": "Prašumska hrčkovica", "shape": "coral", "nc": false}, {"id": "r-psilopezia-nummularialis", "lat": "Psilopezia nummularialis", "hr": "Smeđa novčica", "shape": "coral", "nc": false}, {"id": "r-ptychoverpa-bohemica", "lat": "Ptychoverpa bohemica", "hr": "Češka smrčkovica", "shape": "coral", "nc": false}, {"id": "r-pulveroboletus-hemichrysus", "lat": "Pulveroboletus hemichrysus", "hr": "Sumporasti panjevac", "shape": "coral", "nc": false}, {"id": "r-pulveroboletus-lignicola", "lat": "Pulveroboletus lignicola", "hr": "Smeđi panjevac", "shape": "coral", "nc": false}, {"id": "r-pulvinula-globifera", "lat": "Pulvinula globifera", "hr": "Tropska jastučica", "shape": "coral", "nc": false}, {"id": "r-pulvinula-laeterubra", "lat": "Pulvinula laeterubra", "hr": "Svijetlocrvena jastučica", "shape": "coral", "nc": false}, {"id": "r-pulvinula-ovalispora", "lat": "Pulvinula ovalispora", "hr": "Jajastosporna jastučica", "shape": "coral", "nc": false}, {"id": "r-pycnopeziza-sejournei", "lat": "Pycnopeziza sejournei", "hr": "Smeđa bršljanica", "shape": "coral", "nc": false}, {"id": "r-pycnoporellus-fulgens", "lat": "Pycnoporellus fulgens", "hr": "Sjajna plamenoporka", "shape": "coral", "nc": false}, {"id": "r-pyronema-dubium", "lat": "Pyronema dubium", "hr": "Nejasna gomilica", "shape": "coral", "nc": false}, {"id": "r-resupinatus-vetlinianus", "lat": "Resupinatus vetlinianus", "hr": "Bukova postrančica", "shape": "coral", "nc": false}, {"id": "r-rhodocybe-fallax", "lat": "Rhodocybe fallax", "hr": "Bjelkasta rusoglavka", "shape": "coral", "nc": false}, {"id": "r-rhodoscypha-ovilla", "lat": "Rhodoscypha ovilla", "hr": "Planinska rudočaška", "shape": "coral", "nc": false}, {"id": "r-rigidoporus-crocatus", "lat": "Rigidoporus crocatus", "hr": "Šafranasta tvrdoporka", "shape": "coral", "nc": false}, {"id": "r-rodwayella-sessilis", "lat": "Rodwayella sessilis", "hr": "Hrastova glatkočaška", "shape": "coral", "nc": false}, {"id": "r-russula-alnetorum", "lat": "Russula alnetorum", "hr": "Johina krasnica", "shape": "flat", "nc": false}, {"id": "r-sarcoscypha-macaronesica", "lat": "Sarcoscypha macaronesica", "hr": "Kanarski peharček", "shape": "funnel", "nc": false}, {"id": "r-scleroderma-polyrhizum", "lat": "Scleroderma polyrhizum", "hr": "Zvjezdasta krumpirača", "shape": "ball", "nc": false}, {"id": "r-scutellinia-pilatii", "lat": "Scutellinia pilatii", "hr": "Dugodlaka trepavičarka", "shape": "coral", "nc": false}, {"id": "r-scutellinia-tuberculata", "lat": "Scutellinia tuberculata", "hr": "Sitna trepavičarka", "shape": "coral", "nc": false}, {"id": "r-scytinostroma-galactinum", "lat": "Scytinostroma galactinum", "hr": "Mliječna drvokoška", "shape": "coral", "nc": false}, {"id": "r-scytinostroma-odoratum", "lat": "Scytinostroma odoratum", "hr": "Mirisna drvokoška", "shape": "coral", "nc": false}, {"id": "r-scytinostromella-heterogenea", "lat": "Scytinostromella heterogenea", "hr": "Raznorodna drvokoškica", "shape": "coral", "nc": false}, {"id": "r-skeletocutis-odora", "lat": "Skeletocutis odora", "hr": "Mirisna hrskavičarka", "shape": "coral", "nc": false}, {"id": "r-skeletocutis-papyracea", "lat": "Skeletocutis papyracea", "hr": "Papirasta hrskavičarka", "shape": "coral", "nc": false}, {"id": "r-skeletocutis-stellae", "lat": "Skeletocutis stellae", "hr": "Dugovječna hrskavičarka", "shape": "coral", "nc": false}, {"id": "r-sparassis-spathulata", "lat": "Sparassis spathulata", "hr": "Lopatasta kokica", "shape": "coral", "nc": false}, {"id": "r-spongipellis-delectans", "lat": "Spongipellis delectans", "hr": "Žuteći spužvičar", "shape": "coral", "nc": false}, {"id": "r-steccherinum-subcrinale", "lat": "Steccherinum subcrinale", "hr": "Kuštrava bodljarica", "shape": "coral", "nc": false}, {"id": "r-stropharia-dorsipora", "lat": "Stropharia dorsipora", "hr": "Konjska strnišnica", "shape": "convex", "nc": false}, {"id": "r-strossmayeria-rackii", "lat": "Strossmayeria rackii", "hr": "Slavonski lončić", "shape": "coral", "nc": false}, {"id": "r-suillus-lakei", "lat": "Suillus lakei", "hr": "Duglazijina slinavka", "shape": "convex", "nc": false}, {"id": "r-suillus-tridentinus", "lat": "Suillus tridentinus", "hr": "Narančasta slinavka", "shape": "convex", "nc": false}, {"id": "r-tatraea-dumbirensis", "lat": "Tatraea dumbirensis", "hr": "Prašumska planinka", "shape": "coral", "nc": false}, {"id": "r-trichoglossum-confusum", "lat": "Trichoglossum confusum", "hr": "Blijedotrusni dlakojezičnjak", "shape": "coral", "nc": false}, {"id": "r-trichoglossum-hirsutum", "lat": "Trichoglossum hirsutum", "hr": "Veliki dlakojezičnjak", "shape": "coral", "nc": false}, {"id": "r-trichoglossum-variabile", "lat": "Trichoglossum variabile", "hr": "Promjenjivi dlakojezičnjak", "shape": "coral", "nc": false}, {"id": "r-tricholoma-aurantium", "lat": "Tricholoma aurantium", "hr": "Narančasta kružoliska", "shape": "convex", "nc": false}, {"id": "r-tricholoma-caligatum", "lat": "Tricholoma caligatum", "hr": "Krokodilasta kružoliska", "shape": "convex", "nc": false}, {"id": "r-tricholoma-colossus", "lat": "Tricholoma colossus", "hr": "Golema kružoliska", "shape": "convex", "nc": false}, {"id": "r-tricholoma-goniospermum", "lat": "Tricholoma goniospermum", "hr": "Uglatosporna kružoliska", "shape": "convex", "nc": false}, {"id": "r-tricholoma-luridum", "lat": "Tricholoma luridum", "hr": "Sivolistićava kružoliska", "shape": "convex", "nc": false}, {"id": "r-tulostoma-fimbriatum", "lat": "Tulostoma fimbriatum", "hr": "Resasta pušnica", "shape": "coral", "nc": false}, {"id": "r-tulostoma-melanocyclum", "lat": "Tulostoma melanocyclum", "hr": "Tamnousta pušnica", "shape": "coral", "nc": false}, {"id": "r-urnula-craterium", "lat": "Urnula craterium", "hr": "Crni pehar", "shape": "coral", "nc": false}, {"id": "r-veluticeps-ambigua", "lat": "Veluticeps ambigua", "hr": "Smeđa krastavka", "shape": "coral", "nc": false}, {"id": "r-verpa-conica", "lat": "Verpa conica", "hr": "Prstasta smrčkovica", "shape": "tall", "nc": false}, {"id": "r-vibrissea-truncorum", "lat": "Vibrissea truncorum", "hr": "Narančasta nitoglavka", "shape": "coral", "nc": false}, {"id": "r-xerocomus-moravicus", "lat": "Xerocomus moravicus", "hr": "Moravski baršunovac", "shape": "convex", "nc": false}, {"id": "r-xerula-caussei", "lat": "Xerula caussei", "hr": "Crnosmeđa korjenjača", "shape": "convex", "nc": false}, {"id": "p-psilocybe-semilanceata", "lat": "Psilocybe semilanceata", "hr": "Šiljasta golokapica", "shape": "conical", "nc": false}, {"id": "p-psilocybe-serbica", "lat": "Psilocybe serbica", "hr": "Psilocybe serbica", "shape": "convex", "nc": false}, {"id": "p-psilocybe-cyanescens", "lat": "Psilocybe cyanescens", "hr": "Psilocybe cyanescens", "shape": "convex", "nc": false}, {"id": "p-panaeolus-cinctulus", "lat": "Panaeolus cinctulus", "hr": "Panaeolus cinctulus", "shape": "conical", "nc": false}, {"id": "x-chlorophyllum-olivieri", "lat": "Chlorophyllum olivieri", "hr": "", "nc": true}]')
ISPRAVCI = {
 'Lactarius contorversus': 'Lactarius controversus', 'Hygrocybe intermedius': 'Hygrocybe intermedia',
 'Hygrocybe coccineus': 'Hygrocybe coccinea', 'Hygrocybe laeta': 'Gliophorus laetus', 'Camarophyllus niveus': 'Cuphophyllus virgineus',
 'Cantharellus cinerus': 'Craterellus cinereus', 'Boletus albidus': 'Caloboletus radicans', 'Xerocomus parasticus': 'Pseudoboletus parasiticus',
 'Leccinum criseum': 'Leccinum pseudoscabrum', 'Polyporus frandosus': 'Grifola frondosa', 'Polyporus intybaceus': 'Grifola frondosa',
 'Polyporus pes-caprae': 'Albatrellus pes-caprae', 'Calodon aurantiacum': 'Hydnellum aurantiacum', 'Calodon compactum': 'Hydnellum compactum',
 'Dryodon erinaceus': 'Hericium erinaceus', 'Pleurodon auriscalpium': 'Auriscalpium vulgare', 'Clavaria aurea': 'Ramaria aurea',
 'Clavaria pallida': 'Ramaria pallida', 'Clavaria fennica': 'Ramaria fennica', 'Clavaria pistillaris': 'Clavariadelphus pistillaris',
 'Clavaria truncata': 'Clavariadelphus truncatus', 'Tremellodon gelatinosum': 'Pseudohydnum gelatinosum', 'Exidia galndulosa': 'Exidia glandulosa',
 'Mitrophora hybrida': 'Morchella semilibera', 'Lepiota naucina': 'Leucoagaricus leucothites', 'Collybia platyphylla': 'Megacollybia platyphylla',
 'Dermocybe anthracina': 'Cortinarius anthracinus', 'Tricholoma georgi': 'Calocybe gambosa', 'Tricholoma tigrinum': 'Tricholoma pardinum',
 'Leucopaxillus amarus': 'Leucopaxillus gentianeus', 'Entoloma mougeoti': 'Entoloma mougeotii', 'Psathyrella spadiceo-grisea': 'Psathyrella spadiceogrisea',
 'Russula heterophila': 'Russula heterophylla', 'Amanita aspera': 'Amanita franchetii', 'Clitocybe gilva': 'Paralepista gilva',
 'Hygrophorus hypotheius': 'Hygrophorus hypothejus',
}
print(len(TREBA), 'vrsta treba fotografiju')

In [ ]:
def autor_iz(attr):
    a = attr or ''
    m = re.search(r'\(c\)\s*(.+?),\s*(some|all|no) rights', a)
    if m: return m.group(1).strip()
    m = re.search(r'^(.+?),\s*no known copyright', a)
    return (m.group(1) if m else a).strip() or 'nepoznat autor'

def takson(lat):
    t = TAKS.get(lat)
    if t: return t['id']
    q = ISPRAVCI.get(lat, lat)
    for path in ('taxa', 'taxa/autocomplete'):
        j = api(path, q=q, per_page=20) or {}
        for r in j.get('results', []):
            if r.get('iconic_taxon_name') not in (None, 'Fungi', 'Protozoa'): continue
            if r.get('name', '').lower() == q.lower() or (r.get('matched_term') or '').lower() == q.lower():
                return r['id']
    return None

def iz_inat(tid):
    t = ((api(f'taxa/{tid}') or {}).get('results') or [None])[0]
    if t:
        for tp in t.get('taxon_photos') or []:
            p = tp.get('photo') or {}
            if p.get('license_code') in LIC_OK and p.get('url'):
                return {'url': p['url'].replace('/square.', '/medium.'), 'autor': autor_iz(p.get('attribution')),
                        'licencija': LIC_TXT[p['license_code']], 'izvor': f"https://www.inaturalist.org/photos/{p['id']}", 'put': 'iNat glavna'}
    j = api('observations', taxon_id=tid, quality_grade='research', photos='true', photo_license=','.join(LIC_OK),
            order_by='votes', order='desc', per_page=10) or {}
    for o in j.get('results', []):
        for p in o.get('photos', []):
            if p.get('license_code') in LIC_OK and p.get('url'):
                return {'url': p['url'].replace('/square.', '/medium.'), 'autor': autor_iz(p.get('attribution')),
                        'licencija': LIC_TXT[p['license_code']], 'izvor': f"https://www.inaturalist.org/observations/{o['id']}", 'put': 'iNat nalaz'}
    return None

CAPI = 'https://commons.wikimedia.org/w/api.php'
LOSE = re.compile(r'micro|spore|basidi|cystid|herbar|drawing|illustrat|plate|tafel|stamp|map|distribution|\.svg|\.tif', re.I)
def iz_commons(lat):
    q = ISPRAVCI.get(lat, lat)
    for k in range(4):
        try:
            r = S.get(CAPI, params={'action': 'query', 'format': 'json', 'generator': 'search', 'gsrnamespace': 6,
                      'gsrsearch': f'"{q}" filetype:bitmap', 'gsrlimit': 15, 'prop': 'imageinfo',
                      'iiprop': 'url|extmetadata', 'iiurlwidth': 640}, timeout=60); time.sleep(1)
            if r.status_code == 200: break
            time.sleep(20 * (k + 1))
        except requests.RequestException: time.sleep(15)
    else: return None
    pages = sorted((r.json().get('query') or {}).get('pages', {}).values(), key=lambda p: p.get('index', 99))
    for p in pages:
        t = p.get('title', '')
        if LOSE.search(t) or q.split()[0].lower() not in t.lower(): continue
        ii = (p.get('imageinfo') or [{}])[0]; em = ii.get('extmetadata') or {}
        lic = (em.get('LicenseShortName', {}).get('value') or '')
        if re.search(r'NC|ND|GFDL$', lic) or not re.search(r'CC0|CC BY|Public domain|PD', lic, re.I): continue
        autor = re.sub('<[^>]+>', '', _html.unescape(em.get('Artist', {}).get('value') or '')).strip() or 'nepoznat autor'
        return {'url': ii.get('thumburl') or ii.get('url'), 'autor': autor[:120], 'licencija': lic,
                'izvor': ii.get('descriptionurl'), 'put': 'Commons'}
    return None

def spremi(url, f):
    for k in range(3):
        try:
            r = S.get(url, timeout=60)
            if r.status_code != 200: time.sleep(5); continue
            im = Image.open(io.BytesIO(r.content)).convert('RGB')
            w, h = im.size; tw = min(w, h * 4 / 3); th = tw * 3 / 4
            im = im.crop(((w - tw) / 2, (h - th) / 2, (w + tw) / 2, (h + th) / 2)).resize((400, 300), Image.LANCZOS)
            im.save(f, 'JPEG', quality=82, optimize=True); return True
        except Exception:
            time.sleep(5)
    return False

In [ ]:
IDX_F = WORK / 'index_novo.json'
IDX = json.loads(IDX_F.read_text()) if IDX_F.exists() else {}
NEMA = []
for i, v in enumerate(TREBA):
    if v['id'] in IDX and (WORK / (v['id'] + '.jpg')).exists(): continue
    tid = takson(v['lat'])
    p = iz_inat(tid) if tid else None
    if not p: p = iz_commons(v['lat'])
    if p and spremi(p['url'], WORK / (v['id'] + '.jpg')):
        IDX[v['id']] = {k: p[k] for k in ('autor', 'licencija', 'izvor', 'put')}
    else:
        NEMA.append(v['lat'])
    if i % 25 == 0 or i == len(TREBA) - 1:
        IDX_F.write_text(json.dumps(IDX, ensure_ascii=False))
        print(f'{i+1}/{len(TREBA)}', v['lat'], '->', p and p['put'])
IDX_F.write_text(json.dumps(IDX, ensure_ascii=False))
print('Pronađeno', len(IDX), '| bez fotografije', len(NEMA)); print(NEMA[:50])

In [ ]:
ZIP = OUT / 'atlas_foto.zip'
with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_STORED) as z:
    z.writestr('index_novo.json', json.dumps(IDX, ensure_ascii=False, indent=0))
    z.writestr('nema.json', json.dumps(NEMA, ensure_ascii=False))
    for k in IDX: z.write(WORK / (k + '.jpg'), 'foto/' + k + '.jpg')
print('✅ Gotovo:', ZIP, round(ZIP.stat().st_size / 1e6, 1), 'MB — pošalji Claudeu datoteku atlas_foto.zip')
try:
    from google.colab import files; files.download(str(ZIP))
except Exception:
    pass